In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd

P9   = '/content/drive/MyDrive/Grad_share/data_share/P9'
DATA = '/content/drive/MyDrive/Grad_share/data_share/데이터셋 모음'

beh = pd.read_csv(f'{P9}/p8s_behavior.csv')
print(beh.shape)
print(beh.columns.tolist())
beh.head(3)

Mounted at /content/drive
(1224, 19)
['question', 'gold', 'prop', 'kside', 'knows', 'A_cb', 'ctx_id', 'ctx_title', 's_wiki_title', 'orig_gold', 'ctx_label', 'ctx_text', 'answer_surface', 'ans_start', 'ans_end', 'wrong_answer', 'answer', 'behavior', 'final_correct']


,question,gold,prop,kside,knows,A_cb,ctx_id,ctx_title,s_wiki_title,orig_gold,ctx_label,ctx_text,answer_surface,ans_start,ans_end,wrong_answer,answer,behavior,final_correct
0,What genre is Kanoon?,"[""Bollywood"", ""HFI"", ""HLFI"", ""Hindi language f...",genre,K+,True,Kanoon is a Bollywood film genre.,16462170,Farz Aur Kanoon,Kanoon,Bollywood,C+,Farz Aur Kanoon Farz aur Kanoon (English: Duty...,Bollywood,88,97,NaN,Action,other,False
1,Who is the author of Free to Choose?,"[""Milton Friedman"", ""Milton Fridman""]",author,K+,True,Milton Friedman,138407,Free to Choose,Free to Choose,Milton Friedman,C+,Free to Choose Free to Choose: A Personal Stat...,Milton Friedman,368,383,NaN,"The authors of ""Free to Choose: A Personal Sta...",other,False
2,What is Prague the capital of?,"[""Central Bohemian Region"", ""Středočeský kraj""...",capital of,K+,True,Czech Republic.,18314033,Coat of arms of Prague,Prague,Kingdom of Bohemia,C+,"of Prague, currently 112. Coat of arms of Prag...",Kingdom of Bohemia,108,122,NaN,The Czech Republic.,retain,True


In [ ]:
# 2x2 셀별 샘플 수
print("=== K x C 셀 크기 ===")
print(pd.crosstab(beh['kside'], beh['ctx_label'], margins=True))

# 셀별 behavior 분포
print("\n=== 셀별 behavior 분포 ===")
print(pd.crosstab([beh['kside'], beh['ctx_label']], beh['behavior'], margins=True))

# 셀별 baseline EM
print("\n=== 셀별 baseline EM ===")
print(beh.groupby(['kside','ctx_label'])['final_correct'].agg(['mean','count']).round(3))

=== K x C 셀 크기 ===
ctx_label   C+   C-   All
kside                    
K+         378  378   756
K−         234  234   468
All        612  612  1224

=== 셀별 behavior 분포 ===
behavior         abstain  accept  other  retain   All
kside ctx_label                                      
K+    C+               0     288     30      60   378
      C-               1     224     48     105   378
K−    C+               0     180     31      23   234
      C-               0     188     33      13   234
All                    1     880    142     201  1224

=== 셀별 baseline EM ===
                  mean  count
kside ctx_label              
K+    C+         0.921    378
      C-         0.458    378
K−    C+         0.868    234
      C-         0.128    234


In [ ]:
# K- 샘플의 A_cb 실제 내용
k_neg = beh[beh['kside']=='K−']
print("K- 샘플 수:", len(k_neg))
print("\n=== A_cb 샘플 20개 (K-) ===")
for i, r in k_neg.sample(20, random_state=0).iterrows():
    print(f"[Q] {r['question'][:60]}")
    print(f"  gold: {r['orig_gold'][:40]}")
    print(f"  A_cb: {str(r['A_cb'])[:80]}")
    print()

# A_cb 길이 분포 (회피성 답변은 대체로 김)
print("=== A_cb 길이 분포 ===")
print(beh.groupby('kside')['A_cb'].apply(lambda s: s.astype(str).str.len().describe()).round(1))

K- 샘플 수: 468

=== A_cb 샘플 20개 (K-) ===
[Q] Who was the producer of The Wild Life?
  gold: Cameron Crowe
  A_cb: I don't have information on the producer of The Wild Life.

[Q] What genre is Roper?
  gold: pop punk
  A_cb: I don't have enough information to determine the genre of "Roper". Could you ple

[Q] Who was the director of Crave?
  gold: Charles de Lauzirika
  A_cb: Michael Haneke

[Q] Who was the screenwriter for P Se PM Tak?
  gold: Kundan Shah
  A_cb: The screenwriter for P Se PM Tak was Sanjay Chouhan.

[Q] Who was the director of The Emerald City?
  gold: Simon Cellan Jones
  A_cb: I couldn't find any information on a film called "The Emerald City". Could you p

[Q] What genre is Alexander Beggrov?
  gold: marine art
  A_cb: I don't have information on Alexander Beggrov's genre. Can you provide more cont

[Q] In what country is Davidson?
  gold: Australia
  A_cb: New Zealand

[Q] What genre is Roper?
  gold: pop punk
  A_cb: I don't have enough information to determine the 

In [ ]:
import re

ABSTAIN_PAT = re.compile(
    r"(i don'?t have|i couldn'?t find|i'?m not sure|i do not have|"
    r"no information|not enough information|can you provide|could you please|"
    r"unable to find|i'?m unable|unfortunately)", re.I)

beh['is_abstain_cb'] = beh['A_cb'].astype(str).str.contains(ABSTAIN_PAT)
beh['cb_type'] = np.where(beh['kside']=='K+', 'K+',
                  np.where(beh['is_abstain_cb'], 'K-_abstain', 'K-_confwrong'))

print("=== cb_type 분포 ===")
print(beh['cb_type'].value_counts())

print("\n=== cb_type x ctx_label : baseline EM ===")
print(beh.groupby(['cb_type','ctx_label'])['final_correct'].agg(['mean','count']).round(3))

print("\n=== cb_type x ctx_label : behavior 분포 ===")
print(pd.crosstab([beh['cb_type'], beh['ctx_label']], beh['behavior'], normalize='index').round(3))

print("\n=== A_cb 길이 (검증용) ===")
print(beh.groupby('cb_type')['A_cb'].apply(lambda s: s.astype(str).str.len().median()))

=== cb_type 분포 ===
cb_type
K+              756
K-_confwrong    294
K-_abstain      174
Name: count, dtype: int64

=== cb_type x ctx_label : baseline EM ===
                         mean  count
cb_type      ctx_label              
K+           C+         0.921    378
             C-         0.458    378
K-_abstain   C+         0.908     87
             C-         0.103     87
K-_confwrong C+         0.844    147
             C-         0.143    147

=== cb_type x ctx_label : behavior 분포 ===
behavior                abstain  accept  other  retain
cb_type      ctx_label                                
K+           C+           0.000   0.762  0.079   0.159
             C-           0.003   0.593  0.127   0.278
K-_abstain   C+           0.000   0.770  0.092   0.138
             C-           0.000   0.908  0.057   0.034
K-_confwrong C+           0.000   0.769  0.156   0.075
             C-           0.000   0.741  0.190   0.068

=== A_cb 길이 (검증용) ===
cb_type
K+               13.0
K-_abstain  

/tmp/ipykernel_3720/2878616889.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  beh['is_abstain_cb'] = beh['A_cb'].astype(str).str.contains(ABSTAIN_PAT)


In [ ]:
# accept 개입 타깃 셀
tgt = beh[(beh['cb_type']=='K-_confwrong') & (beh['ctx_label']=='C+')]
print("타깃 셀 크기:", len(tgt), "| other:", (tgt['behavior']=='other').sum())

print("\n=== other 응답 실제 내용 (15개) ===")
for i, r in tgt[tgt['behavior']=='other'].head(15).iterrows():
    print(f"[Q] {r['question'][:55]}")
    print(f"  ctx답(정답): {r['answer_surface'][:40]}")
    print(f"  A_cb(파라메트릭): {str(r['A_cb'])[:50]}")
    print(f"  실제출력    : {str(r['answer'])[:70]}")
    print()

# 회복 상한 계산
print("=== accept 개입 상한 (other가 전부 accept로 전환 시) ===")
for grp in ['K-_confwrong','K-_abstain','K+']:
    s = beh[(beh['cb_type']==grp) & (beh['ctx_label']=='C+')]
    base, other_r = s['final_correct'].mean(), (s['behavior']=='other').mean()
    print(f"{grp:14s} n={len(s):4d}  base={base:.3f}  →상한={base+other_r:.3f}  (여지 +{other_r:.3f})")

타깃 셀 크기: 147 | other: 23

=== other 응답 실제 내용 (15개) ===
[Q] What is the religion of Thomas Cooray?
  ctx답(정답): Catholic Church
  A_cb(파라메트릭): Roman Catholicism
  실제출력    : Roman Catholic

[Q] Who was the screenwriter for The Duellists?
  ctx답(정답): Joseph Conrad
  A_cb(파라메트릭): David Lean and Robert Bolt
  실제출력    : Unfortunately, the text does not mention the screenwriter's name. Howe

[Q] What is Jay the capital of?
  ctx답(정답): Delaware County
  A_cb(파라메트릭): Jay is the capital of Delaware.
  실제출력    : The Huckleberry Capital of the World.

[Q] Who was the director of Postal?
  ctx답(정답): Uwe Boll
  A_cb(파라메트릭): Vipul Mehta
  실제출력    : The question doesn't explicitly mention the director of the movie "Pos

[Q] Who was the screenwriter for Return of the Fly?
  ctx답(정답): Edward Bernds
  A_cb(파라메트릭): Mark Rochelle
  실제출력    : Unfortunately, the text does not mention the screenwriter for "Return 

[Q] What genre is Raavan?
  ctx답(정답): action film
  A_cb(파라메트릭): Fantasy
  실제출력    : The genre o

In [ ]:
tgt = beh[(beh['cb_type']=='K-_confwrong') & (beh['ctx_label']=='C+')].copy()

# 1) span 위치 검증
def span_ok(r):
    try:
        s, e = int(r['ans_start']), int(r['ans_end'])
        return str(r['ctx_text'])[s:e].strip() == str(r['answer_surface']).strip()
    except: return False

tgt['span_ok'] = tgt.apply(span_ok, axis=1)
tgt['surf_in_ctx'] = tgt.apply(
    lambda r: str(r['answer_surface']).lower() in str(r['ctx_text']).lower(), axis=1)

print("=== span 정합성 (K-_confwrong × C+, n=147) ===")
print(f"ans_start/end 일치 : {tgt['span_ok'].mean():.3f}  ({tgt['span_ok'].sum()}/{len(tgt)})")
print(f"단순 문자열 포함   : {tgt['surf_in_ctx'].mean():.3f}  ({tgt['surf_in_ctx'].sum()}/{len(tgt)})")

# 2) prop별 behavior — 어떤 질문 유형이 깨지는지
print("\n=== prop별 other 비율 ===")
p = tgt.groupby('prop')['behavior'].agg(
    n='size', other=lambda s:(s=='other').mean(), accept=lambda s:(s=='accept').mean()).round(3)
print(p.sort_values('other', ascending=False))

# 3) '문맥에 답 없다'고 응답한 케이스 탐지
NOMENT = r"(does ?n'?t mention|does not mention|not mentioned|doesn'?t explicitly|"\
         r"does not explicitly|no mention|not provide|does not provide|doesn'?t state|does not state)"
tgt['says_absent'] = tgt['answer'].astype(str).str.contains(NOMENT, case=False, regex=True)
print("\n=== 모델이 '문맥에 답 없음'이라 응답한 비율 ===")
print(pd.crosstab(tgt['behavior'], tgt['says_absent']))

=== span 정합성 (K-_confwrong × C+, n=147) ===
ans_start/end 일치 : 0.844  (124/147)
단순 문자열 포함   : 0.871  (128/147)

=== prop별 other 비율 ===
                 n  other  accept
prop                             
capital of       5  0.600   0.000
religion         7  0.429   0.429
genre            7  0.429   0.429
producer        24  0.333   0.625
screenwriter    12  0.250   0.667
capital          5  0.200   0.800
director        22  0.045   0.909
author          23  0.043   0.913
father          12  0.000   0.917
composer        12  0.000   0.917
country          3  0.000   1.000
place of birth   8  0.000   1.000
occupation       4  0.000   1.000
mother           1  0.000   1.000
sport            2  0.000   0.500

=== 모델이 '문맥에 답 없음'이라 응답한 비율 ===
says_absent  False  True 
behavior                 
accept         103     10
other            9     14
retain          11      0


/tmp/ipykernel_3720/4125581022.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  tgt['says_absent'] = tgt['answer'].astype(str).str.contains(NOMENT, case=False, regex=True)


## Strat 로딩

In [ ]:
import os
SV_PATH = "/content/drive/MyDrive/Grad_share/data_share/P6/strat_v2_behavior.parquet"
sv = pd.read_parquet(SV_PATH)

print(sv.shape)
print(sv.columns.tolist())
print()
display(sv.head(3))

# 저카디널리티 범주형 컬럼 = 데이터셋명/kside/ctx_label 후보
print("=== 범주형 컬럼 ===")
for c in sv.columns:
    if sv[c].dtype == object and sv[c].nunique() < 15:
        print(f"[{c}] n={sv[c].nunique()} : {sv[c].value_counts().to_dict()}")

# A-2 / (b) 가능 여부
print("\n=== 필요 컬럼 존재 여부 ===")
for key in ['A_cb','logprob','prob','conf','entropy','score','knows','kside','ctx_label','behavior','final_correct','dataset','source']:
    hit = [c for c in sv.columns if key.lower() in c.lower()]
    print(f"{key:14s} → {hit if hit else '없음'}")

(1916, 43)
['sample_id', 'dataset_id', 'fact_nature', 'construction_operator', 'reasoning_structure', 'provenance', 'desired_action', 'world_truth', 'context_internal_validity', 'benchmark_target', 'question', 'context', 'parametric_answer', 'contextual_answer', 'relation_type', 'answer_type', 'hop_count', 'domain', 'mod_degree_bin', 'num_edits', 's_pop', 'mulan_date', 'answer_candidates', 'cb_generation', 'K_state3', 'K', 'ans_ok', 'context_trunc', 'ans_preserved', 'prompt_len', 'cb_samples', 'n_known5', 'instability', 'refusal_rate', 'greedy_refusal', 'is_abstain_group', 'cb_samples_str', 'greedy_known_v2', 'open_book_gen', 'n_steps', 'behavior', 'no_conflict', 'hedged']



,sample_id,dataset_id,fact_nature,construction_operator,reasoning_structure,provenance,desired_action,world_truth,context_internal_validity,benchmark_target,...,refusal_rate,greedy_refusal,is_abstain_group,cb_samples_str,greedy_known_v2,open_book_gen,n_steps,behavior,no_conflict,hedged
0,clasheval_1902,clasheval,static,numeric_perturbation,single_hop,semi_synthetic,reject,false,coherent,parametric,...,0.0,False,False,2016 ||| 2016 ||| 2016 ||| 2016 ||| 2016,True,"The year Phil Knight's memoir, 'Shoe Dog', was...",37,both,False,False
1,clasheval_2491,clasheval,static,numeric_perturbation,single_hop,semi_synthetic,reject,false,coherent,parametric,...,0.0,False,False,1944 ||| 1944 ||| 1944 ||| 1944 ||| 1944,True,"The USS Missouri (BB-63), an Iowa-class battle...",29,both,False,False
2,clasheval_1258,clasheval,static,numeric_perturbation,single_hop,semi_synthetic,reject,false,coherent,parametric,...,0.4,False,False,2009 ||| 2009 ||| I couldn't find any informat...,True,1954.,4,accepted_context,False,False


=== 범주형 컬럼 ===
[dataset_id] n=4 : {'clasheval': 480, 'confiqa_qa': 480, 'dynamicqa': 479, 'nqswap': 477}
[fact_nature] n=3 : {'static': 1670, 'temporal_mutable': 200, 'disputable': 46}
[construction_operator] n=5 : {'entity_swap': 620, 'relation_edit': 480, 'numeric_perturbation': 337, 'no_perturbation': 279, 'temporal_replacement': 200}
[reasoning_structure] n=1 : {'single_hop': 1916}
[provenance] n=2 : {'semi_synthetic': 1436, 'synthetic': 480}
[desired_action] n=3 : {'accept': 957, 'reject': 480, 'needs_verification': 479}
[world_truth] n=3 : {'false': 1670, 'outdated': 200, 'viewpoint_dependent': 46}
[context_internal_validity] n=1 : {'coherent': 1916}
[benchmark_target] n=3 : {'contextual': 957, 'parametric': 480, 'abstention': 479}
[answer_type] n=2 : {'entity': 1102, 'numeric': 337}
[domain] n=9 : {'static': 233, 'temporal': 200, 'drugs': 130, 'locations': 83, 'years': 77, 'news': 73, 'names': 60, 'records': 57, 'disputable': 46}
[mod_degree_bin] n=4 : {'small': 177, 'medium': 1

TypeError: unhashable type: 'numpy.ndarray'

In [ ]:
# 1) 핵심 컬럼 실제 값 확인
for c in ['K','K_state3','desired_action','behavior','greedy_known_v2','is_abstain_group','no_conflict','hedged','ans_ok']:
    print(f"[{c}] {sv[c].value_counts(dropna=False).to_dict()}")

# 2) 데이터셋 × desired_action
print("\n=== dataset_id × desired_action ===")
print(pd.crosstab(sv['dataset_id'], sv['desired_action']))

# 3) 데이터셋 × K × behavior
print("\n=== dataset_id × K × behavior ===")
print(pd.crosstab([sv['dataset_id'], sv['K']], sv['behavior']))

# 4) confidence 프록시 분포
print("\n=== instability / refusal_rate / n_known5 ===")
print(sv.groupby('dataset_id')[['instability','refusal_rate','n_known5']].describe().T.round(3))

# 5) cb_generation 샘플
print("\n=== cb_generation 예시 ===")
for d in sv['dataset_id'].unique():
    r = sv[sv['dataset_id']==d].iloc[0]
    print(f"\n--{d}--")
    print(f" Q  : {str(r['question'])[:60]}")
    print(f" para: {str(r['parametric_answer'])[:40]} | ctx: {str(r['contextual_answer'])[:40]}")
    print(f" cb  : {str(r['cb_generation'])[:70]}")

[K] {'K-': 968, 'K+': 948}
[K_state3] {'K+': 948, 'K-_other': 810, 'K-_refusal': 158}
[desired_action] {'accept': 957, 'reject': 480, 'needs_verification': 479}
[behavior] {'accepted_context': 846, 'both': 352, 'retained_parametric': 323, 'other': 264, 'abstain': 131}
[greedy_known_v2] {False: 992, True: 924}
[is_abstain_group] {False: 1801, True: 115}
[no_conflict] {False: 1916}
[hedged] {False: 1789, True: 127}
[ans_ok] {True: 1916}

=== dataset_id × desired_action ===
desired_action  accept  needs_verification  reject
dataset_id                                        
clasheval            0                   0     480
confiqa_qa         480                   0       0
dynamicqa            0                 479       0
nqswap             477                   0       0

=== dataset_id × K × behavior ===
behavior       abstain  accepted_context  both  other  retained_parametric
dataset_id K                                                              
clasheval  K+       15           

In [ ]:
# both / other의 실제 내용, sf 라벨러 재현 근거 확인
print("=== behavior='both' 예시 5개 (confiqa) ===")
for i,r in sv[(sv['dataset_id']=='confiqa_qa')&(sv['behavior']=='both')].head(5).iterrows():
    print(f"Q: {str(r['question'])[:50]}")
    print(f"  para:{str(r['parametric_answer'])[:30]} | ctx:{str(r['contextual_answer'])[:30]}")
    print(f"  out :{str(r['open_book_gen'])[:90]}\n")

# K 라벨이 어떻게 정의됐는지 역추적
print("=== K vs greedy_known_v2 vs n_known5 ===")
print(pd.crosstab(sv['K'], sv['greedy_known_v2']))
print()
print(sv.groupby('K')[['n_known5','instability','refusal_rate']].mean().round(3))

=== behavior='both' 예시 5개 (confiqa) ===
Q: What country was Franklin Delano Roosevelt a citiz
  para:United States of America | ctx:Germany
  out :The statement contains an error. Franklin Delano Roosevelt was the 32nd President of the U

Q: Who directed the film First Man?
  para:Damien Chazelle | ctx:Stephen Gaghan
  out :The film First Man was directed by Damien Chazelle, not Stephen Gaghan. Damien Chazelle di

Q: Where is the headquarters of the Minnesota Vikings
  para:Minnesota | ctx:Redmond
  out :I think there may be a slight issue here.  The team is actually called the Minnesota Vikin

Q: What band is Louise Redknapp a member of?
  para:Eternal | ctx:Jawbreakers
  out :The question seems to be referring to Louise Redknapp, not Louise from the R&B group Jawbr

Q: What organization does George Soros employ?
  para:Open Society Foundations | ctx:Didi Hirsch Mental Health Serv
  out :The information provided does not accurately describe George Soros' work or the organizati

=== K 

In [ ]:
import re
def norm(s):
    s = str(s).lower().strip()
    return re.sub(r'[^a-z0-9 ]','',s)

def contains(out, ans):
    o, a = norm(out), norm(ans)
    return len(a)>0 and a in o

d = sv[sv['dataset_id']!='dynamicqa'].copy()   # needs_verification 제외

# sf 라벨러: cb_generation과 open_book 출력 비교로 retain 판정 (gold 미사용)
d['sf_para_in_out'] = d.apply(lambda r: contains(r['open_book_gen'], r['cb_generation']), axis=1)
d['sf_ctx_in_out']  = d.apply(lambda r: contains(r['open_book_gen'], r['contextual_answer']), axis=1)
d['sf_retain'] = d['sf_para_in_out'] & ~d['sf_ctx_in_out']
d['sf_retain_incl_both'] = d['sf_para_in_out']   # both 포함 버전

# gold retain: behavior 기준
d['gold_retain']      = d['behavior'].isin(['retained_parametric'])
d['gold_retain_both'] = d['behavior'].isin(['retained_parametric','both'])

print("=== A-2 : gold-free retain 순도 (절대 개수 병기) ===")
for name, sf_col, gold_col in [
    ("both 제외", 'sf_retain', 'gold_retain'),
    ("both 포함", 'sf_retain_incl_both', 'gold_retain_both')]:
    print(f"\n--- {name} ---")
    rows=[]
    for ds, g in d.groupby('dataset_id'):
        tp = (g[sf_col] & g[gold_col]).sum(); pred = g[sf_col].sum(); act = g[gold_col].sum()
        rows.append(dict(dataset=ds, n=len(g),
            purity=round(tp/pred,3) if pred else np.nan, tp=tp, pred=pred,
            recall=round(tp/act,3) if act else np.nan, gold_retain=act))
    print(pd.DataFrame(rows).to_string(index=False))

=== A-2 : gold-free retain 순도 (절대 개수 병기) ===

--- both 제외 ---
   dataset   n  purity  tp  pred  recall  gold_retain
 clasheval 480   0.970  32    33   0.508           63
confiqa_qa 480   0.730  46    63   0.590           78
    nqswap 477   0.738  45    61   0.433          104

--- both 포함 ---
   dataset   n  purity  tp  pred  recall  gold_retain
 clasheval 480   0.862  50    58   0.526           95
confiqa_qa 480   0.822 139   169   0.675          206
    nqswap 477   0.793  88   111   0.458          192


In [ ]:
# sf가 놓친 gold retain (FN) 실제 내용
print("=== FN 예시: gold=retain인데 sf가 못 잡은 것 ===")
fn = d[d['gold_retain_both'] & ~d['sf_retain_incl_both']]
print("FN 총:", len(fn), "| 데이터셋별:", fn['dataset_id'].value_counts().to_dict())
for ds in ['nqswap','confiqa_qa']:
    print(f"\n--{ds}--")
    for i,r in fn[fn['dataset_id']==ds].head(4).iterrows():
        print(f" cb  : {str(r['cb_generation'])[:45]}")
        print(f" out : {str(r['open_book_gen'])[:75]}")
        print(f" beh : {r['behavior']}\n")

# FP 예시: sf가 retain이라 했는데 아닌 것
print("=== FP 예시 ===")
fp = d[~d['gold_retain_both'] & d['sf_retain_incl_both']]
print("FP 총:", len(fp), "|", fp['behavior'].value_counts().to_dict())
for i,r in fp.head(4).iterrows():
    print(f" cb:{str(r['cb_generation'])[:35]} | ctx:{str(r['contextual_answer'])[:25]}")
    print(f" out:{str(r['open_book_gen'])[:75]} | beh:{r['behavior']}\n")

=== FN 예시: gold=retain인데 sf가 못 잡은 것 ===
FN 총: 216 | 데이터셋별: {'nqswap': 104, 'confiqa_qa': 67, 'clasheval': 45}

--nqswap--
 cb  : Winter (June to August)
 out : Based on the information provided, the Western Cape, which includes Cape To
 beh : retained_parametric

 cb  : Hawaii, California, and Tahiti.
 out : The classic surfing movie "Endless Summer" was filmed in various locations 
 beh : both

 cb  : Ernest Rutherford and Niels Bohr
 out : Ernest Rutherford made the nuclear model of an atom.
 beh : retained_parametric

 cb  : Keyshia Cole, and the song was written by Key
 out : The song "Trust and Believe" by Keyshia Cole was written by Keyshia Cole, a
 beh : retained_parametric


--confiqa_qa--
 cb  : Skeeter Sisters, E-Type Boys, and Eternal.
 out : The question seems to be referring to Louise Redknapp, not Louise from the 
 beh : both

 cb  : Russia (officially known as the Russian Feder
 out : The country referred to as "United States of America" in the given context 
 beh : both

In [ ]:
def norm2(s):
    s = re.sub(r'\([^)]*\)','',str(s).lower())          # 괄호 제거
    s = re.sub(r'[^a-z0-9 ]',' ',s)
    return re.sub(r'\s+',' ',s).strip()

STOP = set("the a an of in on at is was were and or not".split())
def toks(s): return [t for t in norm2(s).split() if t not in STOP and len(t)>1]

def soft_match(out, ans, thr=0.6):
    """ans의 토큰 중 thr 이상이 out에 등장하면 매칭"""
    ta, to = toks(ans), set(toks(out))
    if not ta: return False
    # 다중값 대응: 콤마/and 분리 후 하나라도 매칭되면 True
    for part in re.split(r',| and ', str(ans)):
        tp = toks(part)
        if tp and sum(t in to for t in tp)/len(tp) >= thr: return True
    return sum(t in ta_ for t in ta)/len(ta) >= thr if (ta_:=to) else False

d['p2'] = d.apply(lambda r: soft_match(r['open_book_gen'], r['cb_generation']), axis=1)
d['c2'] = d.apply(lambda r: soft_match(r['open_book_gen'], r['contextual_answer']), axis=1)
d['same_ans'] = d.apply(lambda r: soft_match(r['cb_generation'], r['contextual_answer'], 0.9), axis=1)

d['sf2'] = d['p2'] & ~d['same_ans']          # cb==ctx 케이스 제외

print("=== A-2 개선판 (soft match + cb==ctx 제외) ===")
rows=[]
for ds, g in d.groupby('dataset_id'):
    tp=(g['sf2']&g['gold_retain_both']).sum(); pred=g['sf2'].sum(); act=g['gold_retain_both'].sum()
    rows.append(dict(dataset=ds, n=len(g), purity=round(tp/pred,3), tp=tp, pred=pred,
                     recall=round(tp/act,3), gold_retain=act, excluded_same=g['same_ans'].sum()))
print(pd.DataFrame(rows).to_string(index=False))

=== A-2 개선판 (soft match + cb==ctx 제외) ===
   dataset   n  purity  tp  pred  recall  gold_retain  excluded_same
 clasheval 480   0.479  68   142   0.716           95              4
confiqa_qa 480   0.715 173   242   0.840          206              4
    nqswap 477   0.710 130   183   0.677          192              1


In [ ]:
# 채택: exact + both 포함
res=[]
for ds,g in d.groupby('dataset_id'):
    tp=(g['sf_retain_incl_both']&g['gold_retain_both']).sum()
    pred=g['sf_retain_incl_both'].sum(); act=g['gold_retain_both'].sum()
    res.append(dict(dataset=ds, n=len(g), purity=round(tp/pred,3),
                    detail=f"{tp}/{pred}", recall=round(tp/act,3), gold_retain=act))
res.append(dict(dataset='p8s(기존)', n=1224, purity=0.963, detail='-', recall=np.nan, gold_retain=201))
final = pd.DataFrame(res)
print(final.to_string(index=False))
final.to_csv(f'{P9}/A2_purity_by_dataset.csv', index=False)

# 참고: soft match 실패 기록
print("\n[기록] soft match 완화 시 순도: clasheval 0.479 / confiqa 0.715 / nqswap 0.710 → 폐기")

   dataset    n  purity  detail  recall  gold_retain
 clasheval  480   0.862   50/58   0.526           95
confiqa_qa  480   0.822 139/169   0.675          206
    nqswap  477   0.793  88/111   0.458          192
   p8s(기존) 1224   0.963       -     NaN          201

[기록] soft match 완화 시 순도: clasheval 0.479 / confiqa 0.715 / nqswap 0.710 → 폐기


In [ ]:
# p8s 진단 결과 저장
beh[['question','kside','ctx_label','cb_type','behavior','final_correct','A_cb','answer']]\
   .to_csv(f'{P9}/A2_p8s_diag.csv', index=False)

# 셀별 요약
p8s_summary = beh.groupby(['cb_type','ctx_label']).agg(
    n=('final_correct','size'),
    baseline_EM=('final_correct','mean'),
    accept_r=('behavior', lambda s:(s=='accept').mean()),
    retain_r=('behavior', lambda s:(s=='retain').mean()),
    other_r=('behavior', lambda s:(s=='other').mean())).round(3)
p8s_summary.to_csv(f'{P9}/A2_p8s_cell_summary.csv')
print(p8s_summary)

# strat_v2 sf 라벨 결과 (B-2 이후 재사용 대비)
d[['sample_id','dataset_id','K','behavior','sf_retain_incl_both','gold_retain_both']]\
  .to_csv(f'{P9}/A2_sv_sf_labels.csv', index=False)

import os
print("\n저장 완료:", [f for f in os.listdir(P9) if f.startswith('A2_')])

                          n  baseline_EM  accept_r  retain_r  other_r
cb_type      ctx_label                                               
K+           C+         378        0.921     0.762     0.159    0.079
             C-         378        0.458     0.593     0.278    0.127
K-_abstain   C+          87        0.908     0.770     0.138    0.092
             C-          87        0.103     0.908     0.034    0.057
K-_confwrong C+         147        0.844     0.769     0.075    0.156
             C-         147        0.143     0.741     0.068    0.190

저장 완료: ['A2_purity_by_dataset.csv', 'A2_p8s_diag.csv', 'A2_p8s_cell_summary.csv', 'A2_sv_sf_labels.csv']


## GPU 전환

### signal vs front 짝지은 비교

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, os, torch

P9 = "/content/drive/MyDrive/Grad_share/data_share/P9"
pairs = pd.read_csv(f"{P9}/p8s_behavior.csv")
HID   = np.load(f"{P9}/hid_p8s.npy")
print("HID:", HID.shape)
print("중복 question 수:", pairs['question'].duplicated().sum(), "/ unique:", pairs['question'].nunique())

print("\n=== P9 파일 ===")
for f in sorted(os.listdir(P9)): print(" ", f)
print("\nGPU:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HID: (1224, 131072)
중복 question 수: 612 / unique: 612

=== P9 파일 ===
  A2_p8s_cell_summary.csv
  A2_p8s_diag.csv
  A2_purity_by_dataset.csv
  A2_sv_sf_labels.csv
  B2_generations.parquet
  B2_results.csv
  B3_generations.parquet
  B3_results.csv
  B3_unconditional.csv
  acc48.npz
  acc_detail.npz
  acc_front.npz
  acc_split.npz
  accept_push.npz
  bench_retain_sweep.csv
  bench_retain_sweep2.csv
  block_sweep_proj.npz
  figs
  goldfree_v2
  hid_p8s.npy
  hid_p8s_ids.npy
  hid_sv_p9prompt.npy
  hid_sv_p9prompt_ids.npy
  p8s_behavior.csv
  retain48.npz
  retain_detail.npz
  retain_ext.npz
  retain_ext16.npz
  retain_front_oof.npz
  retain_na_m3.npy
  retain_oof.npz
  retain_sweep_gen.csv
  steer_na2_ckpt.npz
  steer_sweep.csv
  sv_layer_ckpt.npz
  w_behavior.npy
  w_behavior_na.npy
  w_behavior_na2.npy
  w_behavior_sf.npy
  w_directions.npz

GPU: NVIDIA A100-SXM

In [ ]:
# 1) HID reshape 규칙 확인
print("131072 =", 131072//4096, "layers x 4096")
H = HID.reshape(len(HID), 32, 4096)
print("layer별 norm (layer-major 가정):", np.linalg.norm(H, axis=2).mean(0)[:8].round(1))
H2 = HID.reshape(len(HID), 4096, 32).transpose(0,2,1)
print("layer별 norm (dim-major 가정) :", np.linalg.norm(H2, axis=2).mean(0)[:8].round(1))

# 2) 저장된 w 구조
wd = np.load(f"{P9}/w_directions.npz", allow_pickle=True)
print("\n=== w_directions.npz ===")
for k in wd.files: print(f"  {k}: {np.array(wd[k]).shape}")

# 3) 기존 front 결과
for f in ['retain_front_oof.npz','acc_front.npz','retain_oof.npz','acc48.npz']:
    z = np.load(f"{P9}/{f}", allow_pickle=True)
    print(f"\n=== {f} ===")
    for k in z.files:
        a = np.array(z[k])
        print(f"  {k}: {a.shape} {a.dtype}", (a.item() if a.shape==() else ""))

131072 = 32 layers x 4096
layer별 norm (layer-major 가정): [0.8 1.1 1.7 2.2 2.8 3.6 4.  4.4]
layer별 norm (dim-major 가정) : [32.4 27.5 27.8 28.1 26.5 29.7 35.5 25.8]

=== w_directions.npz ===
  w_main: (32, 4096)
  w_pooled: (32, 4096)

=== retain_front_oof.npz ===
  0: (378,) int64 
  -2: (378,) int64 
  -3: (378,) int64 

=== acc_front.npz ===
  c2: (234,) int64 
  c3: (234,) int64 

=== retain_oof.npz ===
  0: (378,) int64 
  -2: (378,) int64 
  -3: (378,) int64 
  -4: (378,) int64 

=== acc48.npz ===
  0: (234,) int64 
  2: (234,) int64 
  3: (234,) int64 


In [ ]:
# 기존 결과가 EM 이진인지 확인
z = np.load(f"{P9}/retain_oof.npz")
print("retain_oof coef=0 baseline EM:", z['0'].mean().round(3), "| unique:", np.unique(z['0']))
for k in z.files:
    print(f"  coef={k}: EM={z[k].mean():.3f}  net={int((z[k]-z['0']).sum())}")

print()
zf = np.load(f"{P9}/retain_front_oof.npz")
for k in zf.files:
    print(f"  front coef={k}: EM={zf[k].mean():.3f}  net={int((zf[k]-zf['0']).sum())}")

# 셀 인덱스 재현 확인
pairs['kside'] = pairs['kside'].astype(str)
RET = pairs[(pairs.kside=='K+')&(pairs.ctx_label=='C-')]
ACC = pairs[(pairs.kside=='K−')&(pairs.ctx_label=='C+')]
print(f"\nRET(K+C-) n={len(RET)} | ACC(K−C+) n={len(ACC)}")
print("baseline EM  RET:", RET.final_correct.mean().round(3), "| ACC:", ACC.final_correct.mean().round(3))

retain_oof coef=0 baseline EM: 0.458 | unique: [0 1]
  coef=0: EM=0.458  net=0
  coef=-2: EM=0.463  net=2
  coef=-3: EM=0.511  net=20
  coef=-4: EM=0.503  net=17

  front coef=0: EM=0.458  net=0
  front coef=-2: EM=0.463  net=2
  front coef=-3: EM=0.476  net=7

RET(K+C-) n=378 | ACC(K−C+) n=234
baseline EM  RET: 0.458 | ACC: 0.868


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM free:", torch.cuda.mem_get_info()[0]/1e9, "GB")
print("모델 로드됨:", 'model' in dir())

GPU: NVIDIA A100-SXM4-40GB
VRAM free: 41.95713024 GB
모델 로드됨: False


In [ ]:
import json, re
nb = '/mnt/project/P9_steering_val.ipynb'   # 프로젝트 파일

In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

t0 = time.time()
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map="cuda")
model.eval()
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token

print(f"로드 {time.time()-t0:.0f}s")
print("layers:", model.config.num_hidden_layers, "| dim:", model.config.hidden_size)
print("VRAM:", torch.cuda.memory_allocated()/1e9, "GB")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

로드 60s
layers: 32 | dim: 4096
VRAM: 16.06052352 GB


In [ ]:
import numpy as np, pandas as pd, torch, json, re, string, os, gc, time
from tqdm.auto import tqdm
from sklearn.model_selection import GroupKFold

P9 = "/content/drive/MyDrive/Grad_share/data_share/P9"
pairs9 = pd.read_csv(f"{P9}/p8s_behavior.csv")
HID9   = np.load(f"{P9}/hid_p8s.npy")

def normalize(s):
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    return " ".join(s.split())

def match_gold(text, golds):
    t = normalize(text)
    for g in golds:
        if not str(g).strip(): continue
        if re.search(r"\b"+re.escape(normalize(g))+r"\b", t): return True
    return False

def em(ans, gold):
    return int(match_gold(ans, json.loads(gold) if str(gold).startswith("[") else [gold]))

def layer(H, L): return H[:, (L-1)*4096:L*4096]

def diff_of_means(H, mask_a, mask_b):
    Wm = np.zeros((32, 4096), dtype=np.float32)
    for L in range(1, 33):
        HL = layer(H, L)
        d = HL[mask_a].mean(0) - HL[mask_b].mean(0)
        Wm[L-1] = d / (np.linalg.norm(d) + 1e-8)
    return Wm

def build_prompt_str(row):
    return tok.apply_chat_template(
        [{"role":"user","content":f"{row.ctx_text}\n\nQuestion: {row.question}\nAnswer:"}],
        add_generation_prompt=True, tokenize=False)

tok.padding_side = "left"

# ── hook (σ, w 모두 버퍼 → fold마다 교체) ──
sig_t = torch.zeros(32, device=model.device, dtype=torch.bfloat16)
w_t   = torch.zeros((32,4096), device=model.device, dtype=torch.bfloat16)
_hooks, _cfg = [], {"coef":0.0, "layers":[]}

def make_hook(Lidx):
    def hook(mod, inp, out):
        if _cfg["coef"]==0.0 or Lidx not in _cfg["layers"]: return out
        h = out[0] if isinstance(out, tuple) else out
        h = h + _cfg["coef"]*sig_t[Lidx]*w_t[Lidx].to(h.dtype)
        return (h,)+out[1:] if isinstance(out, tuple) else h
    return hook

def attach():
    for lyr in model.model.layers: lyr._forward_hooks.clear()
    _hooks.clear()
    for i in range(32):
        _hooks.append(model.model.layers[i].register_forward_hook(make_hook(i)))

def detach():
    for h in _hooks: h.remove()
    _hooks.clear()

# ── 배치 생성 (w가 동일한 샘플들끼리만 묶임) ──
@torch.no_grad()
def gen_batch(rows, bs=16):
    """rows: DataFrame. 길이순 정렬 후 배치 생성, 원래 순서로 복원"""
    prompts = [build_prompt_str(r) for _, r in rows.iterrows()]
    order = np.argsort([len(p) for p in prompts])
    outs = [None]*len(prompts)
    for s in range(0, len(order), bs):
        sel = order[s:s+bs]
        enc = tok([prompts[i] for i in sel], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(model.device)
        L = enc["input_ids"].shape[1]
        g = model.generate(**enc, max_new_tokens=48, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        for j, i in enumerate(sel):
            outs[i] = tok.decode(g[j, L:], skip_special_tokens=True).strip()
    return outs

BLOCKS = {"front":[8,9,10,11,12], "signal":[14,15,16,17,18]}
CELLS = {
  "retain": (pairs9[(pairs9.kside=="K+")&(pairs9.ctx_label=="C-")].index.values, [-2,-3]),
  "accept": (pairs9[(pairs9.kside=="K−")&(pairs9.ctx_label=="C+")].index.values, [ 2, 3]),
}
print({k:(len(v[0]),v[1]) for k,v in CELLS.items()})
print("em 테스트:", em("The answer is 2005.", '["2005"]'))

{'retain': (378, [-2, -3]), 'accept': (234, [2, 3])}
em 테스트: 1


In [ ]:
import pyarrow as pa, pyarrow.parquet as pq

GEN_PATH = f"{P9}/B2_generations.parquet"

def load_gen():
    if os.path.exists(GEN_PATH):
        df = pd.read_parquet(GEN_PATH)
        print(f"기존 생성 {len(df)}행 로드")
        return df
    return pd.DataFrame(columns=['cond','row_idx','seed','fold','text'])

GEN = load_gen()
DONE = set(GEN['cond'].unique()) if len(GEN) else set()

def save_gen(new_rows):
    """새 생성분을 즉시 디스크에 반영"""
    global GEN
    GEN = pd.concat([GEN, pd.DataFrame(new_rows)], ignore_index=True)
    GEN.to_parquet(GEN_PATH, index=False)

def cond_key(cell, block, coef, seed):
    return f"{cell}|{block}|{coef}|{seed}"

# 이미 끝난 조건 확인
def is_done(cell, block, coef, seed, n_expect):
    k = cond_key(cell, block, coef, seed)
    return (GEN['cond']==k).sum() >= n_expect

print("완료된 조건:", sorted(DONE) if DONE else "없음")

기존 생성 7956행 로드
완료된 조건: ['accept|front|2|0', 'accept|front|2|1', 'accept|front|2|2', 'accept|front|3|0', 'accept|front|3|1', 'accept|front|3|2', 'accept|none|0|0', 'accept|signal|2|0', 'accept|signal|2|1', 'accept|signal|2|2', 'accept|signal|3|0', 'accept|signal|3|1', 'accept|signal|3|2', 'retain|front|-2|0', 'retain|front|-2|1', 'retain|front|-2|2', 'retain|front|-3|0', 'retain|front|-3|1', 'retain|front|-3|2', 'retain|none|0|0', 'retain|signal|-2|0', 'retain|signal|-2|1', 'retain|signal|-2|2', 'retain|signal|-3|0', 'retain|signal|-3|1', 'retain|signal|-3|2']


In [ ]:
SEEDS = [0, 1, 2]

@torch.no_grad()
def run_condition(cell, block_name, coef, seed, bs=16):
    idxs, _ = CELLS[cell]
    k = cond_key(cell, block_name, coef, seed)
    if is_done(cell, block_name, coef, seed, len(idxs)):
        print(f"skip {k}"); return

    sub = pairs9.loc[idxs]
    groups = sub['question'].values
    blk = BLOCKS.get(block_name, [])
    _cfg["coef"], _cfg["layers"] = float(coef), blk

    # 전체 데이터로 fold 분할 (w 추정 소스는 accept/retain 라벨 샘플)
    ar_all = pairs9.behavior.isin(["accept","retain"]).values
    q_all  = pairs9.question.values
    gkf = GroupKFold(5)
    splits = list(gkf.split(np.arange(len(pairs9)), groups=q_all))
    if seed > 0:   # seed별 재분할
        rng = np.random.RandomState(seed)
        perm = rng.permutation(len(pairs9))
        splits = [(perm[tr], perm[te]) for tr, te in
                  gkf.split(np.arange(len(pairs9)), groups=q_all[perm])]

    t0, new = time.time(), []
    for fi, (tr, te) in enumerate(splits):
        te_cell = np.intersect1d(te, idxs)          # 이 fold의 평가 대상
        if len(te_cell)==0: continue

        if coef != 0:
            gi  = tr[ar_all[tr]]                     # train fold의 라벨 샘플만
            lab = pairs9.behavior.values[gi]
            W   = diff_of_means(HID9[gi], lab=="accept", lab=="retain")
            sg  = np.array([layer(HID9[tr], L).std() for L in range(1,33)], np.float32)
            w_t.copy_(torch.tensor(W,  device=model.device, dtype=torch.bfloat16))
            sig_t.copy_(torch.tensor(sg, device=model.device, dtype=torch.bfloat16))

        texts = gen_batch(pairs9.loc[te_cell], bs=bs)
        new += [dict(cond=k, row_idx=int(i), seed=seed, fold=fi, text=t)
                for i, t in zip(te_cell, texts)]
        save_gen(new[-len(te_cell):])                # 배치 단위 즉시 저장

    print(f"{k}: {len(idxs)}건 {time.time()-t0:.0f}s")

# ── 배치 동일성 검증 먼저 ──
attach()
_cfg["coef"], _cfg["layers"] = 0.0, []
smp = pairs9.loc[CELLS['retain'][0][:24]]
t=time.time(); o1 = gen_batch(smp, bs=1);  t1=time.time()-t
t=time.time(); o16= gen_batch(smp, bs=16); t2=time.time()-t
diff = sum(a!=b for a,b in zip(o1,o16))
em1 = np.mean([em(a, r.gold) for a,(_,r) in zip(o1, smp.iterrows())])
em16= np.mean([em(a, r.gold) for a,(_,r) in zip(o16,smp.iterrows())])
print(f"bs1 {t1:.0f}s EM={em1:.3f} | bs16 {t2:.0f}s EM={em16:.3f} | 텍스트불일치 {diff}/24 | 속도 {t1/t2:.1f}x")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


bs1 28s EM=0.625 | bs16 5s EM=0.625 | 텍스트불일치 6/24 | 속도 6.1x


In [ ]:
attach()
t_all = time.time()

for cell, (idxs, coefs) in CELLS.items():
    # baseline (개입 없음, seed/block 무관 → 1회만)
    run_condition(cell, "none", 0, 0, bs=16)
    for seed in SEEDS:
        for bn in BLOCKS:
            for c in coefs:
                run_condition(cell, bn, c, seed, bs=16)

detach()
print(f"\n총 {(time.time()-t_all)/60:.1f}분 | 생성 {len(GEN)}행")
GEN.to_parquet(GEN_PATH, index=False)

retain|none|0|0: 378건 58s
retain|front|-2|0: 378건 84s
retain|front|-3|0: 378건 85s
retain|signal|-2|0: 378건 84s
retain|signal|-3|0: 378건 84s
retain|front|-2|1: 378건 84s
retain|front|-3|1: 378건 84s
retain|signal|-2|1: 378건 84s
retain|signal|-3|1: 378건 84s
retain|front|-2|2: 378건 84s
retain|front|-3|2: 378건 84s
retain|signal|-2|2: 378건 84s
retain|signal|-3|2: 378건 84s
accept|none|0|0: 234건 33s
accept|front|2|0: 234건 59s
accept|front|3|0: 234건 59s
accept|signal|2|0: 234건 56s
accept|signal|3|0: 234건 54s
accept|front|2|1: 234건 59s
accept|front|3|1: 234건 59s
accept|signal|2|1: 234건 56s
accept|signal|3|1: 234건 54s
accept|front|2|2: 234건 59s
accept|front|3|2: 234건 58s
accept|signal|2|2: 234건 55s
accept|signal|3|2: 234건 54s

총 29.7분 | 생성 7956행


In [ ]:
# ── 채점 ──
gold_map = pairs9['gold'].to_dict()
GEN['em'] = [em(t, gold_map[i]) for t, i in zip(GEN['text'], GEN['row_idx'])]
GEN.to_parquet(GEN_PATH, index=False)

def get(cell, block, coef, seed):
    k = cond_key(cell, block, coef, seed)
    g = GEN[GEN['cond']==k].set_index('row_idx')
    return g['em'], g['fold']

# ── cluster bootstrap ──
def cluster_boot(base, cur, groups, B=2000, seed=0):
    rng = np.random.RandomState(seed)
    uq = np.unique(groups); gi = {g:np.where(groups==g)[0] for g in uq}
    d = cur - base
    nets = []
    for _ in range(B):
        pick = rng.choice(uq, len(uq), replace=True)
        idx = np.concatenate([gi[g] for g in pick])
        nets.append(d[idx].sum() * len(d)/len(idx))
    return float(np.mean(nets)), (float(np.percentile(nets,2.5)), float(np.percentile(nets,97.5)))

# ── 결과표 ──
rows = []
for cell, (idxs, coefs) in CELLS.items():
    base_em, _ = get(cell, "none", 0, 0)
    base_em = base_em.sort_index()
    q = pairs9.loc[base_em.index, 'question'].values
    for c in coefs:
        for bn in BLOCKS:
            for seed in SEEDS:
                cur, _ = get(cell, bn, c, seed); cur = cur.sort_index()
                b, u = base_em.values, cur.values
                rec = int(((b==0)&(u==1)).sum()); brk = int(((b==1)&(u==0)).sum())
                m, ci = cluster_boot(b, u, q, seed=seed)
                rows.append(dict(cell=cell, coef=c, block=bn, seed=seed,
                                 baseEM=round(b.mean(),3), EM=round(u.mean(),3),
                                 rec=rec, brk=brk, net=rec-brk,
                                 boot=round(m,1), lo=round(ci[0],1), hi=round(ci[1],1)))
R = pd.DataFrame(rows)
R.to_csv(f"{P9}/B2_results.csv", index=False)
print(R.to_string(index=False))

  cell  coef  block  seed  baseEM    EM  rec  brk  net  boot    lo   hi
retain    -2  front     0   0.616 0.638   17    9    8   8.0  -2.0 17.0
retain    -2  front     1   0.616 0.638   17    9    8   8.1  -2.0 18.0
retain    -2  front     2   0.616 0.638   17    9    8   8.1  -2.0 18.0
retain    -2 signal     0   0.616 0.593   20   29   -9  -8.9 -23.0  5.0
retain    -2 signal     1   0.616 0.593   20   29   -9  -9.0 -22.0  4.0
retain    -2 signal     2   0.616 0.593   20   29   -9  -9.0 -23.0  4.0
retain    -3  front     0   0.616 0.635   21   14    7   7.0  -5.0 19.0
retain    -3  front     1   0.616 0.635   21   14    7   6.9  -4.0 19.0
retain    -3  front     2   0.616 0.635   21   14    7   7.0  -4.0 19.0
retain    -3 signal     0   0.616 0.587   20   31  -11 -11.0 -25.0  3.0
retain    -3 signal     1   0.616 0.587   20   31  -11 -11.0 -25.0  3.0
retain    -3 signal     2   0.616 0.587   20   31  -11 -10.8 -25.0  3.0
accept     2  front     0   0.868 0.863    0    1   -1  -1.0  -3

In [ ]:
# baseline 불일치 원인
b, _ = get("retain","none",0,0); b=b.sort_index()
print("B2 baseline n:", len(b), "EM:", b.mean().round(3))
print("pairs9 final_correct (동일 셀):", pairs9.loc[b.index,'final_correct'].mean().round(3))
print("index 일치:", set(b.index)==set(CELLS['retain'][0]))

# 텍스트 비교
sub = GEN[(GEN['cond']=="retain|none|0|0")].set_index('row_idx').sort_index()
for i in sub.index[:5]:
    print(f"\nQ: {pairs9.loc[i,'question'][:50]}")
    print(f" gold   : {pairs9.loc[i,'gold'][:50]}")
    print(f" 기존   : {str(pairs9.loc[i,'answer'])[:70]}  → {pairs9.loc[i,'final_correct']}")
    print(f" B2생성 : {sub.loc[i,'text'][:70]}  → {sub.loc[i,'em']}")

# seed 분할 확인
for s in SEEDS:
    _, f = get("retain","front",-2,s)
    print(f"seed{s} fold분포:", f.sort_index().values[:20])

B2 baseline n: 378 EM: 0.616
pairs9 final_correct (동일 셀): 0.458
index 일치: True

Q: What genre is Kanoon?
 gold   : ["Bollywood", "HFI", "HLFI", "Hindi language film 
 기존   : Action  → False
 B2생성 : Action film.  → 0

Q: Who is the author of Free to Choose?
 gold   : ["Milton Friedman", "Milton Fridman"]
 기존   : The authors of "Free to Choose: A Personal Statement" are Milton and R  → False
 B2생성 : The authors of "Free to Choose: A Personal Statement" are economists M  → 0

Q: What is Prague the capital of?
 gold   : ["Central Bohemian Region", "Středočeský kraj", "C
 기존   : The Busia District.  → False
 B2생성 : The text does not mention Prague being the capital of a country, but r  → 1

Q: What is Barcelona the capital of?
 gold   : ["Catalonia", "Catalunya", "Cataluña", "Catalonha"
 기존   : The text doesn't explicitly state that Barcelona is the capital of a c  → True
 B2생성 : The text does not explicitly state that Barcelona is the capital of a   → 1

Q: What is the capital of Loir-et-C

In [ ]:
def make_splits(seed, n_splits=5):
    """그룹(question)을 seed로 셔플 후 균등 배분 → 진짜 재분할"""
    q = pairs9.question.values
    uq = np.unique(q)
    rng = np.random.RandomState(seed)
    rng.shuffle(uq)
    fold_of = {g: i % n_splits for i, g in enumerate(uq)}
    fid = np.array([fold_of[x] for x in q])
    return [(np.where(fid!=k)[0], np.where(fid==k)[0]) for k in range(n_splits)]

# 검증
for s in SEEDS:
    sp = make_splits(s)
    te0 = sp[0][1]
    print(f"seed{s} fold0 te 크기 {len(te0)} | 첫 10개 idx {te0[:10]}")
    # 누수 체크
    tr0 = sp[0][0]
    assert len(set(pairs9.question.values[tr0]) & set(pairs9.question.values[te0]))==0, "누수!"
print("\n누수 없음 확인")

seed0 fold0 te 크기 246 | 첫 10개 idx [23 29 33 35 38 47 53 59 65 66]
seed1 fold0 te 크기 246 | 첫 10개 idx [ 4 11 12 14 18 28 35 47 53 63]
seed2 fold0 te 크기 246 | 첫 10개 idx [ 8  9 13 18 20 32 47 52 55 60]

누수 없음 확인


In [ ]:
# 1) 기존 개입 조건 삭제 (baseline만 남김)
GEN = GEN[GEN['cond'].str.contains(r"\|none\|")].reset_index(drop=True)
GEN.to_parquet(GEN_PATH, index=False)
print("남은 행:", len(GEN), "| 조건:", GEN['cond'].unique())

# 2) run_condition 수정판
@torch.no_grad()
def run_condition(cell, block_name, coef, seed, bs=16):
    idxs, _ = CELLS[cell]
    k = cond_key(cell, block_name, coef, seed)
    if (GEN['cond']==k).sum() >= len(idxs):
        print(f"skip {k}"); return

    blk = BLOCKS.get(block_name, [])
    _cfg["coef"], _cfg["layers"] = float(coef), blk
    ar_all = pairs9.behavior.isin(["accept","retain"]).values
    splits = make_splits(seed)

    t0 = time.time()
    for fi, (tr, te) in enumerate(splits):
        te_cell = np.intersect1d(te, idxs)
        if len(te_cell)==0: continue
        if coef != 0:
            gi  = tr[ar_all[tr]]
            lab = pairs9.behavior.values[gi]
            W   = diff_of_means(HID9[gi], lab=="accept", lab=="retain")
            sg  = np.array([layer(HID9[tr], L).std() for L in range(1,33)], np.float32)
            w_t.copy_(torch.tensor(W,  device=model.device, dtype=torch.bfloat16))
            sig_t.copy_(torch.tensor(sg, device=model.device, dtype=torch.bfloat16))
        texts = gen_batch(pairs9.loc[te_cell], bs=bs)
        save_gen([dict(cond=k, row_idx=int(i), seed=seed, fold=fi, text=t)
                  for i, t in zip(te_cell, texts)])
    print(f"{k}: {len(idxs)}건 {time.time()-t0:.0f}s")

# 3) 재실행
attach(); t_all=time.time()
for cell, (idxs, coefs) in CELLS.items():
    for seed in SEEDS:
        for bn in BLOCKS:
            for c in coefs:
                run_condition(cell, bn, c, seed, bs=16)
detach()
print(f"\n총 {(time.time()-t_all)/60:.1f}분 | 생성 {len(GEN)}행")

남은 행: 612 | 조건: ['retain|none|0|0' 'accept|none|0|0']
retain|front|-2|0: 378건 84s
retain|front|-3|0: 378건 83s
retain|signal|-2|0: 378건 83s
retain|signal|-3|0: 378건 83s
retain|front|-2|1: 378건 85s
retain|front|-3|1: 378건 85s
retain|signal|-2|1: 378건 85s
retain|signal|-3|1: 378건 86s
retain|front|-2|2: 378건 83s
retain|front|-3|2: 378건 83s
retain|signal|-2|2: 378건 84s
retain|signal|-3|2: 378건 84s
accept|front|2|0: 234건 58s
accept|front|3|0: 234건 58s
accept|signal|2|0: 234건 56s
accept|signal|3|0: 234건 54s
accept|front|2|1: 234건 57s
accept|front|3|1: 234건 57s
accept|signal|2|1: 234건 56s
accept|signal|3|1: 234건 53s
accept|front|2|2: 234건 58s
accept|front|3|2: 234건 58s
accept|signal|2|2: 234건 55s
accept|signal|3|2: 234건 54s

총 28.0분 | 생성 7956행


In [ ]:
gold_map = pairs9['gold'].to_dict()
GEN['em'] = [em(t, gold_map[i]) for t, i in zip(GEN['text'], GEN['row_idx'])]
GEN.to_parquet(GEN_PATH, index=False)

def get(cell, block, coef, seed):
    g = GEN[GEN['cond']==cond_key(cell,block,coef,seed)].set_index('row_idx').sort_index()
    return g['em'].values, g.index.values

def cluster_boot(base, cur, groups, B=2000, seed=0):
    rng = np.random.RandomState(seed)
    uq = np.unique(groups); gi = {g:np.where(groups==g)[0] for g in uq}
    d = cur - base
    nets = [d[np.concatenate([gi[g] for g in rng.choice(uq,len(uq),replace=True)])].sum()
            *len(d)/len(np.concatenate([gi[g] for g in rng.choice(uq,len(uq),replace=True)])) for _ in range(0)]
    nets=[]
    for _ in range(B):
        pick = rng.choice(uq, len(uq), replace=True)
        idx = np.concatenate([gi[g] for g in pick])
        nets.append(d[idx].sum()*len(d)/len(idx))
    return float(np.mean(nets)), (float(np.percentile(nets,2.5)), float(np.percentile(nets,97.5)))

rows=[]
for cell,(idxs,coefs) in CELLS.items():
    b, bi = get(cell,"none",0,0)
    q = pairs9.loc[bi,'question'].values
    for c in coefs:
        for bn in BLOCKS:
            for s in SEEDS:
                u,_ = get(cell,bn,c,s)
                rec=int(((b==0)&(u==1)).sum()); brk=int(((b==1)&(u==0)).sum())
                m,ci = cluster_boot(b,u,q,seed=s)
                rows.append(dict(cell=cell,coef=c,block=bn,seed=s,baseEM=round(b.mean(),3),
                                 EM=round(u.mean(),3),rec=rec,brk=brk,net=rec-brk,
                                 boot=round(m,1),lo=round(ci[0],1),hi=round(ci[1],1)))
R=pd.DataFrame(rows); R.to_csv(f"{P9}/B2_results.csv",index=False)
print(R.to_string(index=False))

# 짝지은 차이 (signal - front), 동일 seed·동일 샘플
print("\n=== paired: signal - front ===")
for cell,(idxs,coefs) in CELLS.items():
    b,bi = get(cell,"none",0,0); q = pairs9.loc[bi,'question'].values
    for c in coefs:
        for s in SEEDS:
            sg,_ = get(cell,"signal",c,s); fr,_ = get(cell,"front",c,s)
            m,ci = cluster_boot(fr.astype(int), sg.astype(int), q, seed=s)
            print(f"{cell} coef{c:+} seed{s}: diff {m:+.1f} CI[{ci[0]:.0f},{ci[1]:.0f}]")

  cell  coef  block  seed  baseEM    EM  rec  brk  net  boot    lo   hi
retain    -2  front     0   0.616 0.627   15   11    4   3.9  -6.0 14.0
retain    -2  front     1   0.616 0.640   16    7    9   9.1   0.0 19.0
retain    -2  front     2   0.616 0.630   17   12    5   5.1  -5.0 16.0
retain    -2 signal     0   0.616 0.611   21   23   -2  -2.0 -15.0 11.0
retain    -2 signal     1   0.616 0.598   18   25   -7  -7.2 -20.0  5.0
retain    -2 signal     2   0.616 0.616   24   24    0   0.1 -13.0 13.0
retain    -3  front     0   0.616 0.638   24   16    8   7.9  -4.0 21.0
retain    -3  front     1   0.616 0.640   22   13    9   9.0  -2.0 21.0
retain    -3  front     2   0.616 0.627   17   13    4   4.1  -6.0 15.0
retain    -3 signal     0   0.616 0.595   24   32   -8  -8.0 -23.0  6.0
retain    -3 signal     1   0.616 0.587   24   35  -11 -11.1 -26.0  4.0
retain    -3 signal     2   0.616 0.571   21   38  -17 -16.9 -32.0 -2.0
accept     2  front     0   0.868 0.872    2    1    1   1.0  -2

retain에서는 front가 더 좋다는 결론
기존 결과와 반대돼서 검증 필요

In [ ]:
attach()

# (a) 16토큰 baseline — CSV의 0.458이 16토큰 산물인지
idxs = CELLS['retain'][0]
_cfg["coef"], _cfg["layers"] = 0.0, []
sub = pairs9.loc[idxs]
prompts = [build_prompt_str(r) for _,r in sub.iterrows()]
order = np.argsort([len(p) for p in prompts]); o16=[None]*len(prompts)
with torch.no_grad():
    for s in range(0,len(order),16):
        sel=order[s:s+16]
        enc=tok([prompts[i] for i in sel],return_tensors="pt",padding=True,add_special_tokens=False).to(model.device)
        L=enc["input_ids"].shape[1]
        g=model.generate(**enc,max_new_tokens=16,do_sample=False,pad_token_id=tok.pad_token_id)
        for j,i in enumerate(sel): o16[i]=tok.decode(g[j,L:],skip_special_tokens=True).strip()
em16 = np.array([em(t,r.gold) for t,(_,r) in zip(o16,sub.iterrows())])
print(f"[16토큰 baseline] EM={em16.mean():.3f}  (CSV 0.458 / 48토큰 0.616과 대조)")

# (b) w_main(리키) vs OOF — 동일 파이프라인에서 비교
W = np.load(f"{P9}/w_directions.npz"); w_main = W["w_main"].astype(np.float32)
Hk = HID9[idxs]
sg_full = np.array([layer(Hk,L).std() for L in range(1,33)], np.float32)
w_t.copy_(torch.tensor(w_main,device=model.device,dtype=torch.bfloat16))
sig_t.copy_(torch.tensor(sg_full,device=model.device,dtype=torch.bfloat16))

b48,_ = get("retain","none",0,0)
q = pairs9.loc[idxs,'question'].values
print("\n=== w_main(리키, 전체데이터 w) ===")
for bn,blk in BLOCKS.items():
    for c in [-2,-3]:
        _cfg["coef"], _cfg["layers"] = float(c), blk
        txt = gen_batch(sub, bs=16)
        cur = np.array([em(t,r.gold) for t,(_,r) in zip(txt,sub.iterrows())])
        rec=int(((b48==0)&(cur==1)).sum()); brk=int(((b48==1)&(cur==0)).sum())
        m,ci = cluster_boot(b48,cur,q)
        print(f"  {bn:7} coef{c}: EM {cur.mean():.3f} 순증 {rec-brk:+3} CI[{ci[0]:.0f},{ci[1]:.0f}]")
detach()

[16토큰 baseline] EM=0.344  (CSV 0.458 / 48토큰 0.616과 대조)

=== w_main(리키, 전체데이터 w) ===
  front   coef-2: EM 0.632 순증  +6 CI[-6,18]
  front   coef-3: EM 0.622 순증  +2 CI[-10,14]
  signal  coef-2: EM 0.632 순증  +6 CI[-7,19]
  signal  coef-3: EM 0.632 순증  +6 CI[-8,20]


In [ ]:
z = np.load(f"{P9}/retain_oof.npz")
b_old, c_old = z['0'], z['-3']
b_new,_ = get("retain","none",0,0); c_new,_ = get("retain","signal",-3,0)
print(f"기존 OOF: base {b_old.mean():.3f} → signal-3 {c_old.mean():.3f}  순증 {int((c_old-b_old).sum())}")
print(f"B2  OOF: base {b_new.mean():.3f} → signal-3 {c_new.mean():.3f}  순증 {int((c_new-b_new).sum())}")
print(f"\n기존 base vs B2 base 일치: {(b_old==b_new).mean():.3f}")
print(f"기존 base vs CSV final_correct 일치: {(b_old==pairs9.loc[CELLS['retain'][0],'final_correct'].values.astype(int)).mean():.3f}")

기존 OOF: base 0.458 → signal-3 0.511  순증 20
B2  OOF: base 0.616 → signal-3 0.595  순증 -8

기존 base vs B2 base 일치: 0.709
기존 base vs CSV final_correct 일치: 1.000


In [ ]:
acc_idx = CELLS['accept'][0]
for f in ['acc48.npz','acc_detail.npz','acc_split.npz','accept_push.npz','retain48.npz','retain_ext.npz']:
    try:
        z = np.load(f"{P9}/{f}")
        k0 = [k for k in z.files if k.endswith('0') or k=='c0']
        if not k0: print(f"{f}: baseline키 없음 | keys={z.files}"); continue
        for k in k0:
            a = z[k]
            n = len(a)
            csv = pairs9.loc[acc_idx if n==234 else CELLS['retain'][0],'final_correct'].values.astype(int)
            print(f"{f}[{k}] n={n} EM={a.mean():.3f} | CSV일치 {(a==csv).mean():.3f}")
    except Exception as e: print(f"{f}: {e}")

acc48.npz[0] n=234 EM=0.868 | CSV일치 1.000
acc_detail.npz[c0] n=234 EM=0.752 | CSV일치 0.885
acc_split.npz: baseline키 없음 | keys=['oracle_2', 'oracle_3', 'oof_2', 'oof_3']
accept_push.npz: operands could not be broadcast together with shapes (203,) (378,) 
retain48.npz[front_0] n=378 EM=0.458 | CSV일치 1.000
retain48.npz[signal_0] n=378 EM=0.458 | CSV일치 1.000
retain_ext.npz: baseline키 없음 | keys=['-4', '-5', '-8']


In [ ]:
zf = np.load(f"{P9}/retain_front_oof.npz")
csv = pairs9.loc[CELLS['retain'][0],'final_correct'].values.astype(int)
b2,_ = get("retain","none",0,0)
print("retain_front_oof['0'] vs CSV 일치:", (zf['0']==csv).mean())
print("retain_front_oof['0'] vs B2 base 일치:", (zf['0']==b2).mean())

# CSV answer가 어떤 설정인지: 길이 분포
print("\nCSV answer 길이 median:", pairs9['answer'].astype(str).str.len().median())
print("B2 48tok 길이 median  :",
      GEN[GEN['cond']=='retain|none|0|0']['text'].str.len().median())

retain_front_oof['0'] vs CSV 일치: 1.0
retain_front_oof['0'] vs B2 base 일치: 0.708994708994709

CSV answer 길이 median: 44.0
B2 48tok 길이 median  : 120.5


## 다음 실험 _ 무조건 개입

In [ ]:
B3_PATH = f"{P9}/B3_generations.parquet"
G3 = pd.read_parquet(B3_PATH) if os.path.exists(B3_PATH) else \
     pd.DataFrame(columns=['cond','row_idx','seed','fold','text'])
print(f"기존 {len(G3)}행 | 완료조건 {G3['cond'].nunique() if len(G3) else 0}개")

def save3(rows):
    global G3
    G3 = pd.concat([G3, pd.DataFrame(rows)], ignore_index=True)
    G3.to_parquet(B3_PATH, index=False)

# 4셀 정의
CELL4 = {
    "K+C+": pairs9[(pairs9.kside=="K+")&(pairs9.ctx_label=="C+")].index.values,
    "K+C-": pairs9[(pairs9.kside=="K+")&(pairs9.ctx_label=="C-")].index.values,
    "K-C+": pairs9[(pairs9.kside=="K−")&(pairs9.ctx_label=="C+")].index.values,
    "K-C-": pairs9[(pairs9.kside=="K−")&(pairs9.ctx_label=="C-")].index.values,
}
COEFS3 = [2,3,5,8,-2,-3,-5,-8]
print({k:len(v) for k,v in CELL4.items()})

@torch.no_grad()
def run3(cellname, block_name, coef, seed=0, bs=16):
    idxs = CELL4[cellname]
    k = f"{cellname}|{block_name}|{coef}|{seed}"
    if (G3['cond']==k).sum() >= len(idxs):
        print(f"skip {k}"); return
    blk = BLOCKS.get(block_name, [])
    _cfg["coef"], _cfg["layers"] = float(coef), blk
    ar_all = pairs9.behavior.isin(["accept","retain"]).values
    t0 = time.time()
    for fi,(tr,te) in enumerate(make_splits(seed)):
        te_cell = np.intersect1d(te, idxs)
        if len(te_cell)==0: continue
        if coef != 0:
            gi  = tr[ar_all[tr]]
            lab = pairs9.behavior.values[gi]
            W   = diff_of_means(HID9[gi], lab=="accept", lab=="retain")
            sg  = np.array([layer(HID9[tr],L).std() for L in range(1,33)], np.float32)
            w_t.copy_(torch.tensor(W, device=model.device, dtype=torch.bfloat16))
            sig_t.copy_(torch.tensor(sg, device=model.device, dtype=torch.bfloat16))
        txt = gen_batch(pairs9.loc[te_cell], bs=bs)
        save3([dict(cond=k,row_idx=int(i),seed=seed,fold=fi,text=t)
               for i,t in zip(te_cell,txt)])
    print(f"{k}: {len(idxs)}건 {time.time()-t0:.0f}s")

attach(); t_all=time.time()
for cn in CELL4:
    run3(cn, "none", 0)                      # 자체 baseline
    for bn in BLOCKS:
        for c in COEFS3:
            run3(cn, bn, c)
detach()
print(f"\n총 {(time.time()-t_all)/60:.1f}분 | {len(G3)}행")

기존 0행 | 완료조건 0개
{'K+C+': 378, 'K+C-': 378, 'K-C+': 234, 'K-C-': 234}
K+C+|none|0|0: 378건 50s
K+C+|front|2|0: 378건 75s
K+C+|front|3|0: 378건 76s
K+C+|front|5|0: 378건 77s
K+C+|front|8|0: 378건 77s
K+C+|front|-2|0: 378건 79s
K+C+|front|-3|0: 378건 79s
K+C+|front|-5|0: 378건 79s
K+C+|front|-8|0: 378건 81s
K+C+|signal|2|0: 378건 74s
K+C+|signal|3|0: 378건 72s
K+C+|signal|5|0: 378건 72s
K+C+|signal|8|0: 378건 71s
K+C+|signal|-2|0: 378건 81s
K+C+|signal|-3|0: 378건 81s
K+C+|signal|-5|0: 378건 82s
K+C+|signal|-8|0: 378건 83s
K+C-|none|0|0: 378건 57s
K+C-|front|2|0: 378건 83s
K+C-|front|3|0: 378건 83s
K+C-|front|5|0: 378건 83s
K+C-|front|8|0: 378건 83s
K+C-|front|-2|0: 378건 82s
K+C-|front|-3|0: 378건 83s
K+C-|front|-5|0: 378건 83s
K+C-|front|-8|0: 378건 83s
K+C-|signal|2|0: 378건 83s
K+C-|signal|3|0: 378건 82s
K+C-|signal|5|0: 378건 83s
K+C-|signal|8|0: 378건 79s
K+C-|signal|-2|0: 378건 83s
K+C-|signal|-3|0: 378건 83s
K+C-|signal|-5|0: 378건 84s
K+C-|signal|-8|0: 378건 84s
K-C+|none|0|0: 234건 32s
K-C+|front|2|0: 234건 59s
K-

In [ ]:
# 채점 + collapse 탐지
def collapsed(t):
    w = str(t).split()
    if len(w) >= 8 and len(set(w))/len(w) < 0.3: return True
    return re.search(r"(.{2,10})\1{3,}", str(t)) is not None

G3['em']  = [em(t, gold_map[i]) for t, i in zip(G3['text'], G3['row_idx'])]
G3['col'] = [collapsed(t) for t in G3['text']]
G3['len'] = G3['text'].str.len()
G3.to_parquet(B3_PATH, index=False)

def g3(cn, bn, c, s=0):
    x = G3[G3['cond']==f"{cn}|{bn}|{c}|{s}"].set_index('row_idx').sort_index()
    return x

rows=[]
for cn, idxs in CELL4.items():
    B = g3(cn,"none",0); b = B['em'].values
    q = pairs9.loc[B.index,'question'].values
    for bn in BLOCKS:
        for c in COEFS3:
            X = g3(cn,bn,c); u = X['em'].values
            rec=int(((b==0)&(u==1)).sum()); brk=int(((b==1)&(u==0)).sum())
            m,ci = cluster_boot(b,u,q)
            rows.append(dict(cell=cn, block=bn, coef=c, n=len(b),
                baseEM=round(b.mean(),3), EM=round(u.mean(),3),
                rec=rec, brk=brk, net=rec-brk, lo=round(ci[0],1), hi=round(ci[1],1),
                collapse=round(X['col'].mean(),3), meanlen=int(X['len'].mean())))
S = pd.DataFrame(rows); S.to_csv(f"{P9}/B3_results.csv", index=False)

for cn in CELL4:
    print(f"\n{'='*70}\n[{cn}]  baseline EM {S[S.cell==cn].baseEM.iloc[0]:.3f}  n={S[S.cell==cn].n.iloc[0]}")
    print(S[S.cell==cn][['block','coef','EM','rec','brk','net','lo','hi','collapse','meanlen']].to_string(index=False))


[K+C+]  baseline EM 0.915  n=378
 block  coef    EM  rec  brk  net    lo    hi  collapse  meanlen
 front     2 0.905    4    8   -4 -11.0   3.0       0.0       67
 front     3 0.915    6    6    0  -7.0   7.0       0.0       68
 front     5 0.921    8    6    2  -5.0  10.0       0.0       69
 front     8 0.905    9   13   -4 -13.0   6.0       0.0       71
 front    -2 0.918    3    2    1  -3.0   5.0       0.0       67
 front    -3 0.923    4    1    3  -1.0   7.0       0.0       66
 front    -5 0.929    8    3    5  -2.0  12.0       0.0       66
 front    -8 0.934   12    5    7  -1.0  15.0       0.0       67
signal     2 0.929    7    2    5  -0.0  11.0       0.0       60
signal     3 0.934   11    4    7   0.0  15.0       0.0       59
signal     5 0.926   10    6    4  -4.0  12.0       0.0       57
signal     8 0.926   12    8    4  -5.0  13.0       0.0       56
signal    -2 0.907    5    8   -3 -10.0   4.0       0.0       74
signal    -3 0.910    6    8   -2  -9.0   6.0       0.0 

In [ ]:
print("=== 무조건 개입: 4셀 총합 ===")
tot=[]
for bn in BLOCKS:
    for c in COEFS3:
        s = S[(S.block==bn)&(S.coef==c)]
        # 전체 샘플 pooled bootstrap
        b_all, u_all, q_all = [], [], []
        for cn in CELL4:
            B=g3(cn,"none",0); X=g3(cn,bn,c)
            b_all.append(B['em'].values); u_all.append(X['em'].values)
            q_all.append(pairs9.loc[B.index,'question'].values)
        b=np.concatenate(b_all); u=np.concatenate(u_all); q=np.concatenate(q_all)
        m,ci = cluster_boot(b,u,q)
        tot.append(dict(block=bn, coef=c, net=int((u-b).sum()),
                        lo=round(ci[0],1), hi=round(ci[1],1),
                        EM=round(u.mean(),3), baseEM=round(b.mean(),3)))
T=pd.DataFrame(tot).sort_values('net',ascending=False)
T.to_csv(f"{P9}/B3_unconditional.csv",index=False)
print(T.to_string(index=False))

=== 무조건 개입: 4셀 총합 ===
 block  coef  net     lo    hi    EM  baseEM
 front    -5   16   -2.0  36.0 0.683    0.67
 front    -3   16    0.0  32.0 0.683    0.67
 front    -2   14    0.0  27.0 0.681    0.67
signal    -5    8  -17.0  31.0 0.676    0.67
 front    -8    5  -18.0  26.0 0.674    0.67
signal    -2    4  -13.0  21.0 0.673    0.67
signal    -3    0  -21.0  20.0 0.670    0.67
 front     2   -4  -17.0  10.0 0.667    0.67
 front     3   -5  -20.0  11.0 0.666    0.67
 front     5   -6  -24.0  14.0 0.665    0.67
signal     2  -11  -28.0   7.0 0.661    0.67
signal     3  -14  -34.0   6.0 0.658    0.67
 front     8  -33  -56.0 -10.0 0.643    0.67
signal     5  -45  -68.0 -23.0 0.633    0.67
signal    -8  -99 -133.0 -65.0 0.589    0.67
signal     8 -105 -130.0 -79.0 0.584    0.67


In [ ]:
attach()
for c in [4,6,7]:
    run3("K-C+", "signal", c)
    run3("K-C-", "signal", c)   # 안전성도 같이
detach()

G3['em']  = [em(t, gold_map[i]) for t, i in zip(G3['text'], G3['row_idx'])]
G3['col'] = [collapsed(t) for t in G3['text']]
G3.to_parquet(B3_PATH, index=False)

for cn in ["K-C+","K-C-"]:
    B=g3(cn,"none",0); b=B['em'].values; q=pairs9.loc[B.index,'question'].values
    print(f"\n[{cn}] base {b.mean():.3f}")
    for c in [2,3,4,5,6,7,8]:
        X=g3(cn,"signal",c)
        if len(X)==0: continue
        u=X['em'].values
        rec=int(((b==0)&(u==1)).sum()); brk=int(((b==1)&(u==0)).sum())
        m,ci=cluster_boot(b,u,q)
        print(f"  +{c}: EM {u.mean():.3f} net {rec-brk:+3} CI[{ci[0]:.0f},{ci[1]:.0f}]")

K-C+|signal|4|0: 234건 53s
K-C-|signal|4|0: 234건 60s
K-C+|signal|6|0: 234건 51s
K-C-|signal|6|0: 234건 57s
K-C+|signal|7|0: 234건 50s
K-C-|signal|7|0: 234건 57s

[K-C+] base 0.859
  +2: EM 0.872 net  +3 CI[0,7]
  +3: EM 0.872 net  +3 CI[0,7]
  +4: EM 0.880 net  +5 CI[1,10]
  +5: EM 0.885 net  +6 CI[2,11]
  +6: EM 0.876 net  +4 CI[-2,10]
  +7: EM 0.876 net  +4 CI[-1,9]
  +8: EM 0.880 net  +5 CI[0,10]

[K-C-] base 0.192
  +2: EM 0.158 net  -8 CI[-17,0]
  +3: EM 0.137 net -13 CI[-23,-3]
  +4: EM 0.141 net -12 CI[-22,-2]
  +5: EM 0.132 net -14 CI[-24,-5]
  +6: EM 0.115 net -18 CI[-30,-8]
  +7: EM 0.098 net -22 CI[-34,-12]
  +8: EM 0.081 net -26 CI[-38,-14]


## 새로운 런타임 시작

In [ ]:
# ── 1. 마운트 + 로드 ──
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, os, re, json, string, time, torch
from tqdm.auto import tqdm

P9 = "/content/drive/MyDrive/Grad_share/data_share/P9"
pairs9 = pd.read_csv(f"{P9}/p8s_behavior.csv")
HID9   = np.load(f"{P9}/hid_p8s.npy")

GEN = pd.read_parquet(f"{P9}/B2_generations.parquet")
G3  = pd.read_parquet(f"{P9}/B3_generations.parquet")
print("B2:", len(GEN), "행 | B3:", len(G3), "행")
print("B3 조건:", G3['cond'].nunique(), "개")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
B2: 7956 행 | B3: 22212 행
B3 조건: 74 개


In [ ]:
def normalize(s):
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    return " ".join(s.split())

def match_gold(text, golds):
    t = normalize(text)
    for g in golds:
        if not str(g).strip(): continue
        if re.search(r"\b"+re.escape(normalize(g))+r"\b", t): return True
    return False

def em(ans, gold):
    return int(match_gold(ans, json.loads(gold) if str(gold).startswith("[") else [gold]))

def layer(H, L): return H[:, (L-1)*4096:L*4096]

def diff_of_means(H, mask_a, mask_b):
    Wm = np.zeros((32, 4096), dtype=np.float32)
    for L in range(1, 33):
        HL = layer(H, L)
        d = HL[mask_a].mean(0) - HL[mask_b].mean(0)
        Wm[L-1] = d / (np.linalg.norm(d) + 1e-8)
    return Wm

def make_splits(seed, n_splits=5):
    q = pairs9.question.values; uq = np.unique(q)
    rng = np.random.RandomState(seed); rng.shuffle(uq)
    fold_of = {g: i % n_splits for i, g in enumerate(uq)}
    fid = np.array([fold_of[x] for x in q])
    return [(np.where(fid!=k)[0], np.where(fid==k)[0]) for k in range(n_splits)]

def cluster_boot(base, cur, groups, B=2000, seed=0):
    rng = np.random.RandomState(seed)
    uq = np.unique(groups); gi = {g:np.where(groups==g)[0] for g in uq}
    d = cur - base; nets=[]
    for _ in range(B):
        pick = rng.choice(uq, len(uq), replace=True)
        idx = np.concatenate([gi[g] for g in pick])
        nets.append(d[idx].sum()*len(d)/len(idx))
    return float(np.mean(nets)), (float(np.percentile(nets,2.5)), float(np.percentile(nets,97.5)))

def collapsed(t):
    w = str(t).split()
    if len(w)>=8 and len(set(w))/len(w)<0.3: return True
    return re.search(r"(.{2,10})\1{3,}", str(t)) is not None

CELL4 = {
    "K+C+": pairs9[(pairs9.kside=="K+")&(pairs9.ctx_label=="C+")].index.values,
    "K+C-": pairs9[(pairs9.kside=="K+")&(pairs9.ctx_label=="C-")].index.values,
    "K-C+": pairs9[(pairs9.kside=="K−")&(pairs9.ctx_label=="C+")].index.values,
    "K-C-": pairs9[(pairs9.kside=="K−")&(pairs9.ctx_label=="C-")].index.values,
}
gold_map = pairs9['gold'].to_dict()
def g3(cn,bn,c,s=0): return G3[G3['cond']==f"{cn}|{bn}|{c}|{s}"].set_index('row_idx').sort_index()

print("헬퍼 로드 완료 |", {k:len(v) for k,v in CELL4.items()})

헬퍼 로드 완료 | {'K+C+': 378, 'K+C-': 378, 'K-C+': 234, 'K-C-': 234}


In [ ]:
FRONT_L = [8,9,10,11,12]   # 0-indexed decoder layer → w 인덱스 동일
ar_all  = pairs9.behavior.isin(["accept","retain"]).values
lab_all = pairs9.behavior.values

def est_w(idx):
    lab = lab_all[idx]
    return diff_of_means(HID9[idx], lab=="accept", lab=="retain")

# 기준 w: 전체 라벨 샘플
gi_full = np.where(ar_all)[0]
W_full  = est_w(gi_full)
print("전체 라벨 샘플:", len(gi_full), "(accept/retain만)")

Ns = [10,20,30,50,75,100,150,200,300,400,600,len(gi_full)]
rows=[]
for N in Ns:
    for sd in range(5):
        rng = np.random.RandomState(sd)
        # question 단위 샘플링 (누수 방지 관성 유지)
        q_uni = np.unique(pairs9.question.values[gi_full])
        rng.shuffle(q_uni)
        pick, cnt = [], 0
        for qq in q_uni:
            sel = gi_full[pairs9.question.values[gi_full]==qq]
            pick.extend(sel); cnt += len(sel)
            if cnt >= N: break
        pick = np.array(pick[:N])
        lab = lab_all[pick]
        if (lab=="accept").sum()<2 or (lab=="retain").sum()<2: continue
        W = est_w(pick)
        cs = [float(np.dot(W[L],W_full[L])) for L in FRONT_L]
        rows.append(dict(N=N, seed=sd, cos_front=np.mean(cs),
                         n_acc=int((lab=="accept").sum()), n_ret=int((lab=="retain").sum())))
C = pd.DataFrame(rows).groupby('N').agg(
    cos=('cos_front','mean'), cos_sd=('cos_front','std'),
    n_acc=('n_acc','mean'), n_ret=('n_ret','mean')).round(3)
print(C.to_string())

전체 라벨 샘플: 1081 (accept/retain만)
        cos  cos_sd  n_acc  n_ret
N                                
10    0.401   0.132    8.0    2.0
20    0.410   0.165   16.6    3.4
30    0.520   0.189   24.8    5.2
50    0.587   0.187   39.6   10.4
75    0.675   0.107   60.8   14.2
100   0.718   0.097   82.8   17.2
150   0.761   0.074  125.8   24.2
200   0.820   0.066  168.2   31.8
300   0.900   0.018  248.2   51.8
400   0.911   0.021  330.2   69.8
600   0.955   0.014  496.0  104.0
1081  1.000   0.000  880.0  201.0


In [ ]:
def est_w_balanced(N, sd):
    """accept/retain 각 N/2씩, question 단위"""
    rng = np.random.RandomState(sd)
    out = []
    for cls in ["accept","retain"]:
        pool = gi_full[lab_all[gi_full]==cls]
        q_uni = np.unique(pairs9.question.values[pool]); rng.shuffle(q_uni)
        pick, cnt = [], 0
        for qq in q_uni:
            sel = pool[pairs9.question.values[pool]==qq]
            pick.extend(sel); cnt += len(sel)
            if cnt >= N//2: break
        out.extend(pick[:N//2])
    return np.array(out)

rows=[]
for N in [10,20,30,50,75,100,150,200,300,400]:
    for sd in range(5):
        pick = est_w_balanced(N, sd)
        lab = lab_all[pick]
        if (lab=="accept").sum()<2 or (lab=="retain").sum()<2: continue
        W = est_w(pick)
        rows.append(dict(N=N, seed=sd,
            cos=np.mean([float(np.dot(W[L],W_full[L])) for L in FRONT_L]),
            n_acc=int((lab=="accept").sum()), n_ret=int((lab=="retain").sum())))
CB = pd.DataFrame(rows).groupby('N').agg(
    cos=('cos','mean'), cos_sd=('cos','std'), n_acc=('n_acc','mean'), n_ret=('n_ret','mean')).round(3)
print("=== 균형 샘플링 ===")
print(CB.to_string())
print("\n=== 비교 (랜덤 vs 균형) ===")
print(pd.DataFrame({'random':C['cos'], 'balanced':CB['cos']}).dropna().round(3).to_string())

=== 균형 샘플링 ===
       cos  cos_sd  n_acc  n_ret
N                               
10   0.184   0.148    5.0    5.0
20   0.316   0.131   10.0   10.0
30   0.416   0.218   15.0   15.0
50   0.611   0.118   25.0   25.0
75   0.650   0.166   37.0   37.0
100  0.714   0.120   50.0   50.0
150  0.812   0.042   75.0   75.0
200  0.881   0.013  100.0  100.0
300  0.922   0.023  150.0  150.0
400  0.950   0.014  200.0  200.0

=== 비교 (랜덤 vs 균형) ===
     random  balanced
N                    
10    0.401     0.184
20    0.410     0.316
30    0.520     0.416
50    0.587     0.611
75    0.675     0.650
100   0.718     0.714
150   0.761     0.812
200   0.820     0.881
300   0.900     0.922
400   0.911     0.950


In [ ]:
# ── hook + 배치 생성 재정의 ──
sig_t = torch.zeros(32, device=model.device, dtype=torch.bfloat16)
w_t   = torch.zeros((32,4096), device=model.device, dtype=torch.bfloat16)
_hooks, _cfg = [], {"coef":0.0, "layers":[]}

def make_hook(Lidx):
    def hook(mod, inp, out):
        if _cfg["coef"]==0.0 or Lidx not in _cfg["layers"]: return out
        h = out[0] if isinstance(out, tuple) else out
        h = h + _cfg["coef"]*sig_t[Lidx]*w_t[Lidx].to(h.dtype)
        return (h,)+out[1:] if isinstance(out, tuple) else h
    return hook

def attach():
    for lyr in model.model.layers: lyr._forward_hooks.clear()
    _hooks.clear()
    for i in range(32): _hooks.append(model.model.layers[i].register_forward_hook(make_hook(i)))
def detach():
    for h in _hooks: h.remove()
    _hooks.clear()

def build_prompt_str(row):
    return tok.apply_chat_template(
        [{"role":"user","content":f"{row.ctx_text}\n\nQuestion: {row.question}\nAnswer:"}],
        add_generation_prompt=True, tokenize=False)

@torch.no_grad()
def gen_batch(rows, bs=16):
    prompts = [build_prompt_str(r) for _, r in rows.iterrows()]
    order = np.argsort([len(p) for p in prompts]); outs=[None]*len(prompts)
    for s in range(0, len(order), bs):
        sel = order[s:s+bs]
        enc = tok([prompts[i] for i in sel], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(model.device)
        L = enc["input_ids"].shape[1]
        g = model.generate(**enc, max_new_tokens=48, do_sample=False, pad_token_id=tok.pad_token_id)
        for j,i in enumerate(sel): outs[i] = tok.decode(g[j,L:], skip_special_tokens=True).strip()
    return outs

# ── A-1 저장소 ──
A1_PATH = f"{P9}/A1_generations.parquet"
GA = pd.read_parquet(A1_PATH) if os.path.exists(A1_PATH) else \
     pd.DataFrame(columns=['cond','row_idx','seed','fold','text'])
def saveA(rows):
    global GA
    GA = pd.concat([GA, pd.DataFrame(rows)], ignore_index=True); GA.to_parquet(A1_PATH, index=False)
print("A1 기존:", len(GA), "행")

A1 기존: 0 행


In [ ]:
RET_IDX = CELL4["K+C-"]
FRONT   = [8,9,10,11,12]
COEF    = -3          # B-3 최적 조건

def pick_N(tr, N, mode, sd):
    """train fold 내에서 N개 샘플 선택 (question 단위)"""
    rng = np.random.RandomState(sd)
    pool = tr[ar_all[tr]]
    if mode == "random":
        q_uni = np.unique(pairs9.question.values[pool]); rng.shuffle(q_uni)
        pick, cnt = [], 0
        for qq in q_uni:
            sel = pool[pairs9.question.values[pool]==qq]
            pick.extend(sel); cnt += len(sel)
            if cnt >= N: break
        return np.array(pick[:N])
    else:  # balanced
        out=[]
        for cls in ["accept","retain"]:
            p = pool[lab_all[pool]==cls]
            q_uni = np.unique(pairs9.question.values[p]); rng.shuffle(q_uni)
            pk, cnt = [], 0
            for qq in q_uni:
                sel = p[pairs9.question.values[p]==qq]
                pk.extend(sel); cnt += len(sel)
                if cnt >= N//2: break
            out.extend(pk[:N//2])
        return np.array(out)

@torch.no_grad()
def runA(N, mode, seed, bs=16):
    k = f"N{N}|{mode}|{seed}"
    if (GA['cond']==k).sum() >= len(RET_IDX): print(f"skip {k}"); return
    _cfg["coef"], _cfg["layers"] = float(COEF), FRONT
    t0=time.time()
    for fi,(tr,te) in enumerate(make_splits(seed)):
        te_cell = np.intersect1d(te, RET_IDX)
        if len(te_cell)==0: continue
        pick = pick_N(tr, N, mode, seed*100+fi) if N>0 else tr[ar_all[tr]]
        lab = lab_all[pick]
        if (lab=="accept").sum()<2 or (lab=="retain").sum()<2: continue
        # 누수 검증
        assert len(set(pairs9.question.values[pick]) & set(pairs9.question.values[te]))==0, "누수!"
        W  = diff_of_means(HID9[pick], lab=="accept", lab=="retain")
        sg = np.array([layer(HID9[tr],L).std() for L in range(1,33)], np.float32)
        w_t.copy_(torch.tensor(W, device=model.device, dtype=torch.bfloat16))
        sig_t.copy_(torch.tensor(sg, device=model.device, dtype=torch.bfloat16))
        txt = gen_batch(pairs9.loc[te_cell], bs=bs)
        saveA([dict(cond=k,row_idx=int(i),seed=seed,fold=fi,text=t) for i,t in zip(te_cell,txt)])
    print(f"{k}: {time.time()-t0:.0f}s")

attach(); t0=time.time()
for N in [30,50,100,200,400]:
    for mode in ["random","balanced"]:
        for sd in [0,1,2]:
            runA(N, mode, sd)
for sd in [0,1,2]:                       # 전체 train (상한)
    runA(0, "full", sd)
detach()
print(f"\n총 {(time.time()-t0)/60:.1f}분 | {len(GA)}행")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


N30|random|0: 84s
N30|random|1: 85s
N30|random|2: 84s
N30|balanced|0: 83s
N30|balanced|1: 86s
N30|balanced|2: 83s
N50|random|0: 82s
N50|random|1: 85s
N50|random|2: 83s
N50|balanced|0: 84s
N50|balanced|1: 86s
N50|balanced|2: 84s
N100|random|0: 83s
N100|random|1: 85s
N100|random|2: 84s
N100|balanced|0: 84s
N100|balanced|1: 86s
N100|balanced|2: 84s
N200|random|0: 82s
N200|random|1: 86s
N200|random|2: 85s
N200|balanced|0: 83s
N200|balanced|1: 86s
N200|balanced|2: 84s
N400|random|0: 84s
N400|random|1: 86s
N400|random|2: 84s
N400|balanced|0: 84s
N400|balanced|1: 86s
N400|balanced|2: 84s
N0|full|0: 83s
N0|full|1: 86s
N0|full|2: 84s

총 46.4분 | 12474행


In [ ]:
GA['em'] = [em(t, gold_map[i]) for t, i in zip(GA['text'], GA['row_idx'])]
GA.to_parquet(A1_PATH, index=False)

B = g3("K+C-","none",0); b = B['em'].values
q = pairs9.loc[B.index,'question'].values
print(f"baseline EM {b.mean():.3f} (n={len(b)})")

rows=[]
for N in [30,50,100,200,400]:
    for mode in ["random","balanced"]:
        for sd in [0,1,2]:
            X = GA[GA['cond']==f"N{N}|{mode}|{sd}"].set_index('row_idx').sort_index()
            if len(X)==0: continue
            u = X['em'].values
            m,ci = cluster_boot(b,u,q,seed=sd)
            rows.append(dict(N=N, mode=mode, seed=sd, EM=round(u.mean(),3),
                net=int((u-b).sum()), lo=round(ci[0],1), hi=round(ci[1],1)))
for sd in [0,1,2]:
    X = GA[GA['cond']==f"N0|full|{sd}"].set_index('row_idx').sort_index()
    if len(X)==0: continue
    u = X['em'].values; m,ci = cluster_boot(b,u,q,seed=sd)
    rows.append(dict(N=1081, mode="full", seed=sd, EM=round(u.mean(),3),
        net=int((u-b).sum()), lo=round(ci[0],1), hi=round(ci[1],1)))
A = pd.DataFrame(rows); A.to_csv(f"{P9}/A1_saturation.csv", index=False)
print(A.to_string(index=False))

print("\n=== N별 평균 (3 seed) ===")
print(A.groupby(['mode','N'])['net'].agg(['mean','std','min','max']).round(1).to_string())

baseline EM 0.603 (n=378)
   N     mode  seed    EM  net    lo    hi
  30   random     0 0.463  -53 -70.0 -36.0
  30   random     1 0.463  -53 -71.0 -35.0
  30   random     2 0.434  -64 -83.0 -44.0
  30 balanced     0 0.476  -48 -68.0 -29.0
  30 balanced     1 0.452  -57 -76.0 -38.0
  30 balanced     2 0.479  -47 -66.0 -29.0
  50   random     0 0.460  -54 -72.0 -37.0
  50   random     1 0.479  -47 -67.0 -28.0
  50   random     2 0.426  -67 -86.0 -47.0
  50 balanced     0 0.479  -47 -66.0 -28.0
  50 balanced     1 0.442  -61 -80.0 -42.0
  50 balanced     2 0.452  -57 -74.0 -39.0
 100   random     0 0.463  -53 -71.0 -36.0
 100   random     1 0.458  -55 -75.0 -36.0
 100   random     2 0.450  -58 -77.0 -39.0
 100 balanced     0 0.481  -46 -65.0 -28.0
 100 balanced     1 0.479  -47 -66.0 -29.0
 100 balanced     2 0.484  -45 -63.0 -27.0
 200   random     0 0.460  -54 -72.0 -36.0
 200   random     1 0.452  -57 -75.0 -39.0
 200   random     2 0.450  -58 -77.0 -39.0
 200 balanced     0 0.474  -

In [ ]:
# 1) B-3와 A-1 full이 같은 조건인지 직접 대조
X3 = g3("K+C-","front",-3); XA = GA[GA['cond']=="N0|full|0"].set_index('row_idx').sort_index()
print("B3 front-3 EM:", X3['em'].mean().round(3), "| A1 full EM:", XA['em'].mean().round(3))
print("텍스트 일치율:", (X3['text'].values == XA['text'].values).mean())
print("\nB3 예시:", X3['text'].iloc[0][:90])
print("A1 예시:", XA['text'].iloc[0][:90])

# 2) w가 제대로 들어갔는지 — full 조건 재현
tr, te = make_splits(0)[0]
pick = tr[ar_all[tr]]
lab = lab_all[pick]
W = diff_of_means(HID9[pick], lab=="accept", lab=="retain")
print("\nw norm:", np.linalg.norm(W[8:13],axis=1).round(3))
print("n_acc/n_ret:", (lab=="accept").sum(), (lab=="retain").sum())

# 3) _cfg 현재 상태
print("\n_cfg:", _cfg)
print("hook 수:", len(_hooks))

B3 front-3 EM: 0.638 | A1 full EM: 0.468
텍스트 일치율: 0.06084656084656084

B3 예시: Action film.
A1 예시: Action film

w norm: [1. 1. 1. 1. 1.]
n_acc/n_ret: 708 155

_cfg: {'coef': -3.0, 'layers': [8, 9, 10, 11, 12]}
hook 수: 0


In [ ]:
# 개입 없이 baseline 재생성 → B3 baseline과 비교
_cfg["coef"], _cfg["layers"] = 0.0, []
attach()
sub = pairs9.loc[CELL4["K+C-"][:48]]
txt = gen_batch(sub, bs=16)
detach()

B = g3("K+C-","none",0)
old = B['text'].values[:48]
new_em = np.array([em(t,r.gold) for t,(_,r) in zip(txt, sub.iterrows())])
old_em = B['em'].values[:48]
print("옛 baseline EM:", old_em.mean().round(3), "| 새 baseline EM:", new_em.mean().round(3))
print("텍스트 일치:", (np.array(txt)==old).mean())
for i in range(3):
    print(f"\n옛: {old[i][:80]}")
    print(f"새: {txt[i][:80]}")

옛 baseline EM: 0.542 | 새 baseline EM: 0.354
텍스트 일치: 0.0625

옛: Action film.
새: Action

옛: The authors of "Free to Choose: A Personal Statement" are economists Milton and 
새: The authors of "Free to Choose: A Personal Statement" are Milton and Rose D. Fri

옛: The text does not mention Prague being the capital of a country, but rather the 
새: The Busia District.


In [ ]:
_cfg["coef"], _cfg["layers"] = 0.0, []
attach()
sub = pairs9.loc[RET_IDX]
txt = gen_batch(sub, bs=16)
detach()
saveA([dict(cond="A1_base", row_idx=int(i), seed=0, fold=-1, text=t)
       for i,t in zip(RET_IDX, txt)])
GA['em'] = [em(t, gold_map[i]) for t,i in zip(GA['text'], GA['row_idx'])]
GA.to_parquet(A1_PATH, index=False)

bb = GA[GA['cond']=="A1_base"].set_index('row_idx').sort_index()
print(f"A1 자체 baseline EM: {bb['em'].mean():.3f}  (B3 baseline 0.603)")

A1 자체 baseline EM: 0.460  (B3 baseline 0.603)


In [ ]:
import transformers, torch
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("pad_token:", tok.pad_token, tok.pad_token_id)
print("eos_token:", tok.eos_token, tok.eos_token_id)
print("padding_side:", tok.padding_side)
print("model gen config:", model.generation_config)

transformers: 5.15.1
torch: 2.11.0+cu128
pad_token: <|eot_id|> 128009
eos_token: <|eot_id|> 128009
padding_side: left
model gen config: GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "temperature": 0.6,
  "top_p": 0.9
}



In [ ]:
_cfg["coef"], _cfg["layers"] = 0.0, []
attach()
s8 = pairs9.loc[CELL4["K+C-"][:8]]
r1 = gen_batch(s8, bs=8)
r2 = gen_batch(s8, bs=8)
detach()
print("두 번 실행 일치:", sum(a==b for a,b in zip(r1,r2)), "/8")
for a,b in list(zip(r1,r2))[:3]:
    print(f"\n1: {a[:70]}\n2: {b[:70]}")

두 번 실행 일치: 8 /8

1: Action
2: Action

1: The authors of "Free to Choose: A Personal Statement" are Milton and R
2: The authors of "Free to Choose: A Personal Statement" are Milton and R

1: The Busia District.
2: The Busia District.


In [ ]:
bb = GA[GA['cond']=="A1_base"].set_index('row_idx').sort_index()
b = bb['em'].values
q = pairs9.loc[bb.index,'question'].values
print(f"A1 baseline EM {b.mean():.3f}\n")

rows=[]
for N,mode in [(n,m) for n in [30,50,100,200,400] for m in ["random","balanced"]]+[(1081,"full")]:
    for sd in [0,1,2]:
        key = f"N0|full|{sd}" if mode=="full" else f"N{N}|{mode}|{sd}"
        X = GA[GA['cond']==key].set_index('row_idx').sort_index()
        if len(X)==0: continue
        u = X['em'].values
        m_,ci = cluster_boot(b,u,q,seed=sd)
        rows.append(dict(N=N,mode=mode,seed=sd,EM=round(u.mean(),3),
                         net=int((u-b).sum()),lo=round(ci[0],1),hi=round(ci[1],1)))
A = pd.DataFrame(rows); A.to_csv(f"{P9}/A1_saturation.csv",index=False)
print(A.groupby(['mode','N'])[['net','lo','hi']].mean().round(1).to_string())
print("\n=== seed별 변동 ===")
print(A.groupby(['mode','N'])['net'].agg(['mean','std','min','max']).round(1).to_string())

A1 baseline EM 0.460

               net    lo    hi
mode     N                    
balanced 30    3.3  -9.3  15.7
         50   -1.0 -14.0  12.0
         100   8.0  -5.3  21.0
         200   1.7 -11.3  14.7
         400   4.7  -8.7  17.7
full     1081  2.7 -10.7  15.7
random   30   -2.7 -16.7  11.0
         50   -2.0 -15.7  11.3
         100  -1.3 -15.0  11.3
         200  -2.3 -16.3  11.7
         400   3.0 -10.0  15.7

=== seed별 변동 ===
               mean   std  min  max
mode     N                         
balanced 30     3.3   5.5   -3    7
         50    -1.0   7.2   -7    7
         100    8.0   1.0    7    9
         200    1.7   3.5   -2    5
         400    4.7   1.5    3    6
full     1081   2.7   1.5    1    4
random   30    -2.7   6.4  -10    1
         50    -2.0  10.1  -13    7
         100   -1.3   2.5   -4    1
         200   -2.3   2.1   -4    0
         400    3.0   3.6    0    7


In [ ]:
B3R_PATH = f"{P9}/B3_rerun_generations.parquet"
GR = pd.read_parquet(B3R_PATH) if os.path.exists(B3R_PATH) else \
     pd.DataFrame(columns=['cond','row_idx','seed','fold','text'])
def saveR(rows):
    global GR
    GR = pd.concat([GR, pd.DataFrame(rows)], ignore_index=True); GR.to_parquet(B3R_PATH, index=False)

BLOCKS = {"front":[8,9,10,11,12], "signal":[14,15,16,17,18]}

@torch.no_grad()
def runR(cn, bn, coef, seed=0, bs=16):
    idxs = CELL4[cn]; k=f"{cn}|{bn}|{coef}|{seed}"
    if (GR['cond']==k).sum() >= len(idxs): print(f"skip {k}"); return
    _cfg["coef"], _cfg["layers"] = float(coef), BLOCKS.get(bn,[])
    t0=time.time()
    for fi,(tr,te) in enumerate(make_splits(seed)):
        te_cell = np.intersect1d(te, idxs)
        if len(te_cell)==0: continue
        if coef!=0:
            gi=tr[ar_all[tr]]; lab=lab_all[gi]
            W=diff_of_means(HID9[gi], lab=="accept", lab=="retain")
            sg=np.array([layer(HID9[tr],L).std() for L in range(1,33)],np.float32)
            w_t.copy_(torch.tensor(W,device=model.device,dtype=torch.bfloat16))
            sig_t.copy_(torch.tensor(sg,device=model.device,dtype=torch.bfloat16))
        txt=gen_batch(pairs9.loc[te_cell],bs=bs)
        saveR([dict(cond=k,row_idx=int(i),seed=seed,fold=fi,text=t) for i,t in zip(te_cell,txt)])
    print(f"{k}: {time.time()-t0:.0f}s")

attach(); t0=time.time()
for cn, cs in [("K+C-",[0,-3]), ("K-C+",[0,5]), ("K-C-",[0,5])]:
    for c in cs:
        for bn in (["none"] if c==0 else ["front","signal"]):
            runR(cn, bn, c)
detach()
print(f"\n{(time.time()-t0)/60:.1f}분 | {len(GR)}행")

# 채점
GR['em'] = [em(t, gold_map[i]) for t,i in zip(GR['text'], GR['row_idx'])]
GR.to_parquet(B3R_PATH, index=False)
def gr(cn,bn,c,s=0): return GR[GR['cond']==f"{cn}|{bn}|{c}|{s}"].set_index('row_idx').sort_index()

print("\n=== 재현 확인 (자체 baseline) ===")
for cn, c in [("K+C-",-3), ("K-C+",5), ("K-C-",5)]:
    B=gr(cn,"none",0); b=B['em'].values; q=pairs9.loc[B.index,'question'].values
    old_b = g3(cn,"none",0)['em'].mean()
    print(f"\n[{cn}] 이번 base {b.mean():.3f} / B3 base {old_b:.3f}")
    for bn in ["front","signal"]:
        X=gr(cn,bn,c); u=X['em'].values
        m,ci=cluster_boot(b,u,q)
        X3=g3(cn,bn,c); b3=g3(cn,"none",0)['em'].values
        print(f"  {bn:7} coef{c:+}: net {int((u-b).sum()):+3} CI[{ci[0]:.0f},{ci[1]:.0f}]"
              f"   (B3: {int((X3['em'].values-b3).sum()):+3})")

K+C-|none|0|0: 58s
K+C-|front|-3|0: 83s
K+C-|signal|-3|0: 83s
K-C+|none|0|0: 34s
K-C+|front|5|0: 58s
K-C+|signal|5|0: 56s
K-C-|none|0|0: 36s
K-C-|front|5|0: 62s
K-C-|signal|5|0: 56s

8.8분 | 2538행

=== 재현 확인 (자체 baseline) ===

[K+C-] 이번 base 0.455 / B3 base 0.603
  front   coef-3: net  +5 CI[-8,18]   (B3: +13)
  signal  coef-3: net +16 CI[-1,33]   (B3:  -3)

[K-C+] 이번 base 0.872 / B3 base 0.859
  front   coef+5: net  -1 CI[-7,5]   (B3:  +3)
  signal  coef+5: net  +5 CI[-2,12]   (B3:  +6)

[K-C-] 이번 base 0.128 / B3 base 0.192
  front   coef+5: net  -5 CI[-12,2]   (B3:  -5)
  signal  coef+5: net -10 CI[-18,-2]   (B3: -14)


In [ ]:
print("transformers:", transformers.__version__)
print("attn impl:", model.config._attn_implementation)
print("dtype:", next(model.parameters()).dtype)
print("device_map:", model.hf_device_map if hasattr(model,'hf_device_map') else 'single')
print("\ntokenizer class:", type(tok).__name__)
print("chat template hash:", hash(tok.chat_template) if tok.chat_template else None)
# 프롬프트 실제 문자열 확인
p = build_prompt_str(pairs9.iloc[0])
print("\n프롬프트 끝 150자:", repr(p[-150:]))
print("토큰 수:", len(tok(p, add_special_tokens=False)['input_ids']))

transformers: 5.15.1
attn impl: sdpa
dtype: torch.bfloat16
device_map: single

tokenizer class: TokenizersBackend
chat template hash: -7415967283003606641

프롬프트 끝 150자: '"Kondaveeti Simham"" itself was a remake of the 1974\n\nQuestion: What genre is Kanoon?\nAnswer:<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
토큰 수: 201


In [ ]:
# B3 세션에서 저장된 baseline 텍스트 vs 지금 생성
old = g3("K+C-","none",0)
new = gr("K+C-","none",0)
print("일치율:", (old['text'].values == new['text'].values).mean())
print(f"EM: 옛 {old['em'].mean():.3f} / 새 {new['em'].mean():.3f}")

# 길이 분포 — 생성이 일찍 멈추는지
print(f"\n길이 median: 옛 {old['text'].str.len().median():.0f} / 새 {new['text'].str.len().median():.0f}")
print(f"토큰 추정   : 옛 {old['text'].str.split().str.len().median():.0f} / 새 {new['text'].str.split().str.len().median():.0f} 단어")

# 불일치 샘플 3개 직접 비교
d = old['text'].values != new['text'].values
for i in np.where(d)[0][:3]:
    print(f"\nQ: {pairs9.loc[old.index[i],'question'][:50]}")
    print(f" 옛: {old['text'].iloc[i][:100]}")
    print(f" 새: {new['text'].iloc[i][:100]}")

일치율: 0.07142857142857142
EM: 옛 0.603 / 새 0.455

길이 median: 옛 122 / 새 137
토큰 추정   : 옛 20 / 새 22 단어

Q: What genre is Kanoon?
 옛: Action film.
 새: Action

Q: Who is the author of Free to Choose?
 옛: The authors of "Free to Choose: A Personal Statement" are economists Milton and Rose D. Friedman.
 새: The authors of "Free to Choose: A Personal Statement" are Milton and Rose D. Friedman.

Q: What is Prague the capital of?
 옛: The text does not mention Prague being the capital of a country, but rather the "Busia District". Ho
 새: The Busia District.


In [ ]:
# B3 세션이 어떤 버전이었는지 단서 찾기
import subprocess
print(subprocess.run(['pip','list','--format=freeze'],capture_output=True,text=True).stdout[:0] or "")
!pip index versions transformers 2>/dev/null | head -3

# 구버전 설치 테스트 (재시작 필요)
print("\n권장: !pip install transformers==4.46.0 후 런타임 재시작")


transformers (5.16.1)
Available versions: 5.16.1, 5.16.0, 5.15.1, 5.15.0, 5.14.1, 5.14.0, 5.13.1, 5.13.0, 5.12.1, 5.12.0, 5.11.0, 5.10.4, 5.10.2, 5.10.1, 5.9.0, 5.8.1, 5.8.0, 5.7.0, 5.6.2, 5.6.1, 5.6.0, 5.5.4, 5.5.3, 5.5.2, 5.5.1, 5.5.0, 5.4.0, 5.3.0, 5.2.0, 5.1.0, 5.0.0, 4.57.6, 4.57.5, 4.57.4, 4.57.3, 4.57.2, 4.57.1, 4.56.2, 4.56.1, 4.56.0, 4.55.4, 4.55.3, 4.55.2, 4.55.1, 4.55.0, 4.54.1, 4.54.0, 4.53.3, 4.53.2, 4.53.1, 4.53.0, 4.52.4, 4.52.3, 4.52.2, 4.52.1, 4.51.3, 4.51.2, 4.51.1, 4.51.0, 4.50.3, 4.50.2, 4.50.1, 4.50.0, 4.49.0, 4.48.3, 4.48.2, 4.48.1, 4.48.0, 4.47.1, 4.47.0, 4.46.3, 4.46.2, 4.46.1, 4.45.2, 4.45.1, 4.45.0, 4.44.2, 4.44.1, 4.44.0, 4.43.4, 4.43.3, 4.43.2, 4.43.1, 4.43.0, 4.42.4, 4.42.3, 4.42.2, 4.42.1, 4.42.0, 4.41.2, 4.41.1, 4.41.0, 4.40.2, 4.40.1, 4.40.0, 4.39.3, 4.39.2, 4.39.1, 4.39.0, 4.38.2, 4.38.1, 4.38.0, 4.37.2, 4.37.1, 4.37.0, 4.36.2, 4.36.1, 4.36.0, 4.35.2, 4.35.1, 4.35.0, 4.34.1, 4.34.0, 4.33.3, 4.33.2, 4.33.1, 4.33.0, 4.32.1, 4.32.0, 4.31.0, 4.30.2, 4.30.1

In [ ]:
print("모델 경로:", model.config._name_or_path)
print("torch_dtype:", model.config.torch_dtype)

# 가중치 지문
import hashlib
w = model.model.layers[0].self_attn.q_proj.weight
print("L0 q_proj 합:", w.float().sum().item())
print("L0 q_proj[0,:5]:", w[0,:5].float().tolist())

# 임베딩 지문
e = model.model.embed_tokens.weight
print("embed 합:", e.float().sum().item())
print("vocab size:", e.shape[0])

모델 경로: meta-llama/Llama-3.1-8B-Instruct
torch_dtype: torch.bfloat16
L0 q_proj 합: 2.359438896179199
L0 q_proj[0,:5]: [0.005157470703125, -0.029296875, -0.006439208984375, -0.0181884765625, -0.003265380859375]
embed 합: 11911.353515625
vocab size: 128256


In [ ]:
_cfg["coef"], _cfg["layers"] = 0.0, []
attach()
sub = pairs9.loc[CELL4["K+C-"][:64]]
res = {}
for bs in [1, 4, 8, 16, 32]:
    txt = gen_batch(sub, bs=bs)
    e = np.mean([em(t,r.gold) for t,(_,r) in zip(txt, sub.iterrows())])
    res[bs] = (txt, e)
    print(f"bs={bs:2}: EM {e:.3f}")
detach()

print("\n=== bs간 텍스트 일치율 ===")
ks = list(res)
for i in range(len(ks)-1):
    a, b = res[ks[i]][0], res[ks[i+1]][0]
    print(f"bs{ks[i]} vs bs{ks[i+1]}: {np.mean([x==y for x,y in zip(a,b)]):.3f}")
print(f"bs1 vs bs32: {np.mean([x==y for x,y in zip(res[1][0], res[32][0])]):.3f}")

bs= 1: EM 0.375
bs= 4: EM 0.375
bs= 8: EM 0.391
bs=16: EM 0.375
bs=32: EM 0.391

=== bs간 텍스트 일치율 ===
bs1 vs bs4: 0.828
bs4 vs bs8: 0.875
bs8 vs bs16: 0.891
bs16 vs bs32: 0.859
bs1 vs bs32: 0.766


In [ ]:
# B-2 재실행 (현재 커널)
B2R_PATH = f"{P9}/B2_rerun_generations.parquet"
GB = pd.read_parquet(B2R_PATH) if os.path.exists(B2R_PATH) else \
     pd.DataFrame(columns=['cond','row_idx','seed','fold','text'])
def saveB(rows):
    global GB
    GB = pd.concat([GB, pd.DataFrame(rows)], ignore_index=True); GB.to_parquet(B2R_PATH, index=False)

@torch.no_grad()
def runB(cn, bn, coef, seed, bs=16):
    idxs = CELL4[cn]; k=f"{cn}|{bn}|{coef}|{seed}"
    if (GB['cond']==k).sum() >= len(idxs): print(f"skip {k}"); return
    _cfg["coef"], _cfg["layers"] = float(coef), BLOCKS.get(bn,[])
    t0=time.time()
    for fi,(tr,te) in enumerate(make_splits(seed)):
        tc=np.intersect1d(te,idxs)
        if len(tc)==0: continue
        if coef!=0:
            gi=tr[ar_all[tr]]; lab=lab_all[gi]
            W=diff_of_means(HID9[gi],lab=="accept",lab=="retain")
            sg=np.array([layer(HID9[tr],L).std() for L in range(1,33)],np.float32)
            w_t.copy_(torch.tensor(W,device=model.device,dtype=torch.bfloat16))
            sig_t.copy_(torch.tensor(sg,device=model.device,dtype=torch.bfloat16))
        saveB([dict(cond=k,row_idx=int(i),seed=seed,fold=fi,text=t)
               for i,t in zip(tc, gen_batch(pairs9.loc[tc],bs=bs))])
    print(f"{k}: {time.time()-t0:.0f}s")

attach(); t0=time.time()
runB("K+C-","none",0,0)
for sd in [0,1,2]:
    for bn in ["front","signal"]:
        for c in [-2,-3]:
            runB("K+C-",bn,c,sd)
detach()
print(f"\n{(time.time()-t0)/60:.1f}분")

K+C-|none|0|0: 58s
K+C-|front|-2|0: 83s
K+C-|front|-3|0: 83s
K+C-|signal|-2|0: 83s
K+C-|signal|-3|0: 83s
K+C-|front|-2|1: 86s
K+C-|front|-3|1: 86s
K+C-|signal|-2|1: 86s
K+C-|signal|-3|1: 87s
K+C-|front|-2|2: 84s
K+C-|front|-3|2: 84s
K+C-|signal|-2|2: 84s
K+C-|signal|-3|2: 84s

17.9분


In [ ]:
GB['em'] = [em(t, gold_map[i]) for t,i in zip(GB['text'], GB['row_idx'])]
GB.to_parquet(B2R_PATH, index=False)
def gb(bn,c,s=0): return GB[GB['cond']==f"K+C-|{bn}|{c}|{s}"].set_index('row_idx').sort_index()

B = gb("none",0,0); b = B['em'].values; q = pairs9.loc[B.index,'question'].values
print(f"baseline EM {b.mean():.3f} (n={len(b)})\n")

rows=[]
for c in [-2,-3]:
    for bn in ["front","signal"]:
        for s in [0,1,2]:
            u = gb(bn,c,s)['em'].values
            m,ci = cluster_boot(b,u,q,seed=s)
            rows.append(dict(coef=c,block=bn,seed=s,EM=round(u.mean(),3),
                net=int((u-b).sum()),lo=round(ci[0],1),hi=round(ci[1],1)))
R2=pd.DataFrame(rows);

baseline EM 0.455 (n=378)



In [ ]:
print(R2.to_string(index=False))
print("\n=== paired: signal - front ===")
for c in [-2,-3]:
    for s in [0,1,2]:
        f_,g_ = gb("front",c,s)['em'].values, gb("signal",c,s)['em'].values
        m,ci = cluster_boot(f_,g_,q,seed=s)
        print(f"coef{c} seed{s}: diff {m:+.1f} CI[{ci[0]:.0f},{ci[1]:.0f}]")

 coef  block  seed    EM  net    lo   hi
   -2  front     0 0.476    8  -3.0 19.0
   -2  front     1 0.458    1  -9.0 11.0
   -2  front     2 0.471    6  -5.0 18.0
   -2 signal     0 0.463    3 -13.0 18.0
   -2 signal     1 0.458    1 -15.0 17.0
   -2 signal     2 0.460    2 -14.0 17.0
   -3  front     0 0.468    5  -8.0 18.0
   -3  front     1 0.463    3 -11.0 15.0
   -3  front     2 0.471    6  -7.0 19.0
   -3 signal     0 0.497   16  -1.0 33.0
   -3 signal     1 0.495   15  -2.0 32.0
   -3 signal     2 0.497   16  -1.0 33.0

=== paired: signal - front ===
coef-2 seed0: diff -5.2 CI[-22,11]
coef-2 seed1: diff -0.0 CI[-16,15]
coef-2 seed2: diff -4.1 CI[-19,11]
coef-3 seed0: diff +11.0 CI[-6,28]
coef-3 seed1: diff +12.1 CI[-5,30]
coef-3 seed2: diff +9.9 CI[-6,26]


In [ ]:
import os
# 1) 결과표 저장 확인
for f in ['A1_saturation.csv','A2_purity_by_dataset.csv','B2_results.csv',
          'B2_rerun_results.csv','B3_results.csv','B3_unconditional.csv']:
    p=f"{P9}/{f}"; print(f"{'OK ' if os.path.exists(p) else '없음'} {f}")

# 2) B3 재현 결과 저장 (아직 안 됨)
rows=[]
for cn,c in [("K+C-",-3),("K-C+",5),("K-C-",5)]:
    B=gr(cn,"none",0); b=B['em'].values; q_=pairs9.loc[B.index,'question'].values
    for bn in ["front","signal"]:
        u=gr(cn,bn,c)['em'].values
        m,ci=cluster_boot(b,u,q_)
        rows.append(dict(cell=cn,block=bn,coef=c,baseEM=round(b.mean(),3),
            EM=round(u.mean(),3),net=int((u-b).sum()),lo=round(ci[0],1),hi=round(ci[1],1)))
pd.DataFrame(rows).to_csv(f"{P9}/B3_rerun_results.csv",index=False)

# 3) 환경 지문 기록 (다음 세션 대조용)
import transformers
env = dict(transformers=transformers.__version__, torch=torch.__version__,
           model=model.config._name_or_path, attn=model.config._attn_implementation,
           embed_sum=float(model.model.embed_tokens.weight.float().sum()),
           qproj_sum=float(model.model.layers[0].self_attn.q_proj.weight.float().sum()),
           base_KpCm=float(gb("none",0,0)['em'].mean()), date="2026-08-30")
pd.DataFrame([env]).to_csv(f"{P9}/ENV_fingerprint_0830.csv",index=False)
print("\n환경 지문:", env)

# 4) 생성물 최종 확인
for nm,df in [("B2",GEN),("B3",G3),("B3_rerun",GR),("B2_rerun",GB),("A1",GA)]:
    print(f"{nm}: {len(df)}행")

OK  A1_saturation.csv
OK  A2_purity_by_dataset.csv
OK  B2_results.csv
없음 B2_rerun_results.csv
OK  B3_results.csv
OK  B3_unconditional.csv

환경 지문: {'transformers': '5.15.1', 'torch': '2.11.0+cu128', 'model': 'meta-llama/Llama-3.1-8B-Instruct', 'attn': 'sdpa', 'embed_sum': 11911.353515625, 'qproj_sum': 2.359438896179199, 'base_KpCm': 0.455026455026455, 'date': '2026-08-30'}
B2: 7956행
B3: 22212행
B3_rerun: 2538행
B2_rerun: 4914행
A1: 12852행


/tmp/ipykernel_1877/2581417398.py:22: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  embed_sum=float(model.model.embed_tokens.weight.float().sum()),
